In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Lakeflow Jobs Orchestration Deployment
# Cell 1 — Lakeflow Jobs config + workspace/API readiness
# ─────────────────────────────────────────────────────────────

import json
import re
import time
import uuid
import platform
import requests
from datetime import datetime, timezone
from urllib.parse import urlparse

from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────
# Product / deployment config
# ─────────────────────────────────────────────────────────────

PROJECT_NAME = "SupplySage AI"
WORKFLOW_NAME = "supplysage_lakeflow_daily_refresh"
WORKFLOW_DISPLAY_NAME = "SupplySage AI - Lakeflow Daily Refresh"
WORKFLOW_VERSION = "v1_lakeflow_jobs_orchestration"

GOLD_SCHEMA = "supplysage_gold"
SILVER_SCHEMA = "supplysage_silver"
BRONZE_SCHEMA = "supplysage_bronze"

CREATED_BY = "Vigya"
NOTEBOOK_NAME = "30_lakeflow_jobs_orchestration_deployment"

# Safety flag:
# Keep this False until we explicitly decide to create/update the Databricks job.
CREATE_OR_UPDATE_JOB = False

# Schedule:
# Daily at 6:00 AM America/Chicago.
# Databricks Jobs uses Quartz cron syntax.
JOB_TIMEZONE_ID = "America/Chicago"
JOB_QUARTZ_CRON = "0 0 6 * * ?"

# Job-level parameters that downstream notebooks can read via widgets.
JOB_BASE_PARAMETERS = {
    "project_name": PROJECT_NAME,
    "workflow_name": WORKFLOW_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "run_mode": "production",
    "refresh_scope": "daily",
    "external_ingestion_mode": "incremental",
    "agent_eval_mode": "smoke_test",
    "serving_health_mode": "health_only"
}

# Default retry / timeout policies.
DEFAULT_MAX_RETRIES = 1
DEFAULT_MIN_RETRY_INTERVAL_MILLIS = 5 * 60 * 1000
DEFAULT_TIMEOUT_SECONDS = 60 * 60

# ─────────────────────────────────────────────────────────────
# Orchestration output tables
# ─────────────────────────────────────────────────────────────

WORKFLOW_JOB_MANIFEST_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_manifest"
WORKFLOW_RUN_STATUS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_run_status"
WORKFLOW_TASK_HEALTH_TABLE = f"{GOLD_SCHEMA}.gold_workflow_task_health"
WORKFLOW_DEPLOYMENT_SUMMARY_TABLE = f"{GOLD_SCHEMA}.gold_workflow_deployment_summary"

# Product tables that prove the system is deployable
REQUIRED_PRODUCT_TABLES = [
    f"{BRONZE_SCHEMA}.bronze_ext_ingestion_batch_manifest",
    f"{GOLD_SCHEMA}.gold_supplier_risk_scores",
    f"{GOLD_SCHEMA}.gold_sku_stockout_risk_scores",
    f"{GOLD_SCHEMA}.gold_rag_embeddings",
    f"{GOLD_SCHEMA}.gold_rag_retrieval_index",
    f"{GOLD_SCHEMA}.gold_chat_context_snapshots",
    f"{GOLD_SCHEMA}.gold_supplier_risk_agent_monitoring_metrics",
    f"{GOLD_SCHEMA}.gold_supplier_risk_agent_eval_results",
    f"{GOLD_SCHEMA}.gold_supplier_risk_agent_serving_health",
    f"{GOLD_SCHEMA}.gold_supplier_risk_agent_deployment_manifest"
]

# ─────────────────────────────────────────────────────────────
# Workspace context helpers
# ─────────────────────────────────────────────────────────────

def get_context_value(name: str):
    """
    Safely get Databricks notebook context values.
    """
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()

        if name == "notebook_path":
            return ctx.notebookPath().get()

        if name == "browser_host_name":
            return ctx.browserHostName().get()

        if name == "api_url":
            return ctx.apiUrl().get()

        if name == "workspace_id":
            return ctx.workspaceId().get()

        if name == "user":
            return ctx.userName().get()

    except Exception:
        return None

    return None


CURRENT_NOTEBOOK_PATH = get_context_value("notebook_path")
WORKSPACE_HOST = get_context_value("browser_host_name")
WORKSPACE_API_URL = get_context_value("api_url")
WORKSPACE_ID = get_context_value("workspace_id")
CURRENT_USER = get_context_value("user")

if CURRENT_NOTEBOOK_PATH:
    NOTEBOOK_BASE_PATH = "/".join(CURRENT_NOTEBOOK_PATH.split("/")[:-1])
else:
    NOTEBOOK_BASE_PATH = "/Users/vigya.awasthi@tamu.edu"

print("Workspace context:")
print("CURRENT_NOTEBOOK_PATH:", CURRENT_NOTEBOOK_PATH)
print("NOTEBOOK_BASE_PATH:", NOTEBOOK_BASE_PATH)
print("WORKSPACE_HOST:", WORKSPACE_HOST)
print("WORKSPACE_API_URL:", WORKSPACE_API_URL)
print("WORKSPACE_ID:", WORKSPACE_ID)
print("CURRENT_USER:", CURRENT_USER)

# ─────────────────────────────────────────────────────────────
# Notebook path registry
# ─────────────────────────────────────────────────────────────
# Assumption:
# These notebooks live in the same workspace folder as Notebook 30.
# If your notebooks are in a different folder, update NOTEBOOK_BASE_PATH once.

NOTEBOOK_PATHS = {
    # Bronze external ingestion and validation
    "05_bronze_ingest_external_risk_sources_v0": f"{NOTEBOOK_BASE_PATH}/05_bronze_ingest_external_risk_sources_v0",
    "06_bronze_validate_external_risk_sources": f"{NOTEBOOK_BASE_PATH}/06_bronze_validate_external_risk_sources",

    # Silver layer
    "07_silver_m5_calendar": f"{NOTEBOOK_BASE_PATH}/07_silver_m5_calendar",
    "08_silver_m5_sell_prices": f"{NOTEBOOK_BASE_PATH}/08_silver_m5_sell_prices",
    "09_silver_m5_sales_daily": f"{NOTEBOOK_BASE_PATH}/09_silver_m5_sales_daily",
    "10_silver_retail_inventory": f"{NOTEBOOK_BASE_PATH}/10_silver_retail_inventory",
    "11_silver_dataco_orders_shipments": f"{NOTEBOOK_BASE_PATH}/11_silver_dataco_orders_shipments",
    "12_silver_synthetic_supplier_tables": f"{NOTEBOOK_BASE_PATH}/12_silver_synthetic_supplier_tables",
    "13_silver_external_risk_events_evidence": f"{NOTEBOOK_BASE_PATH}/13_silver_external_risk_events_evidence",
    "14_silver_relationship_validation": f"{NOTEBOOK_BASE_PATH}/14_silver_relationship_validation",
    "15_silver_domain_views": f"{NOTEBOOK_BASE_PATH}/15_silver_domain_views",

    # Gold marts
    "16_gold_dim_suppliers": f"{NOTEBOOK_BASE_PATH}/16_gold_dim_suppliers",
    "17_gold_dim_products_skus": f"{NOTEBOOK_BASE_PATH}/17_gold_dim_products_skus",
    "18_gold_supplier_sku_dependency_mart": f"{NOTEBOOK_BASE_PATH}/18_gold_supplier_sku_dependency_mart",
    "19_gold_inventory_stockout_feature_mart": f"{NOTEBOOK_BASE_PATH}/19_gold_inventory_stockout_feature_mart",
    "20_gold_supplier_performance_mart": f"{NOTEBOOK_BASE_PATH}/20_gold_supplier_performance_mart",
    "21_gold_external_risk_event_mart": f"{NOTEBOOK_BASE_PATH}/21_gold_external_risk_event_mart",
    "22_gold_supplier_risk_scores": f"{NOTEBOOK_BASE_PATH}/22_gold_supplier_risk_scores",
    "23_gold_sku_stockout_risk_scores": f"{NOTEBOOK_BASE_PATH}/23_gold_sku_stockout_risk_scores",
    "24_27_gold_rag_alerts_dashboards": f"{NOTEBOOK_BASE_PATH}/24_27_gold_rag_alerts_dashboards",

    # RAG / context / agent
    "25_gold_rag_embeddings": f"{NOTEBOOK_BASE_PATH}/25_gold_rag_embeddings",
    "25b_gold_chat_context_snapshots": f"{NOTEBOOK_BASE_PATH}/25b_gold_chat_context_snapshots",
    "26_rag_retrieval_test": f"{NOTEBOOK_BASE_PATH}/26_rag_retrieval_test",
    "27_langgraph_supplier_risk_agent": f"{NOTEBOOK_BASE_PATH}/27_langgraph_supplier_risk_agent",

    # Monitoring / serving
    "28_mlflow_agent_monitoring_evaluation": f"{NOTEBOOK_BASE_PATH}/28_mlflow_agent_monitoring_evaluation",
    "29_agent_serving_deployment_wrapper": f"{NOTEBOOK_BASE_PATH}/29_agent_serving_deployment_wrapper",
    "30_lakeflow_jobs_orchestration_deployment": f"{NOTEBOOK_BASE_PATH}/30_lakeflow_jobs_orchestration_deployment"
}

print("\nRegistered notebook paths:")
for name, path in NOTEBOOK_PATHS.items():
    print(f"{name}: {path}")

# ─────────────────────────────────────────────────────────────
# Lakeflow Jobs task graph
# ─────────────────────────────────────────────────────────────

LAKEFLOW_TASK_GRAPH = [
    {
        "task_key": "bronze_external_ingestion",
        "description": "Ingest external risk sources into Bronze Delta tables.",
        "notebook_key": "05_bronze_ingest_external_risk_sources_v0",
        "depends_on": [],
        "layer": "bronze",
        "critical": True
    },
    {
        "task_key": "bronze_external_validation",
        "description": "Validate latest clean Bronze ingestion batch.",
        "notebook_key": "06_bronze_validate_external_risk_sources",
        "depends_on": ["bronze_external_ingestion"],
        "layer": "bronze",
        "critical": True
    },
    {
        "task_key": "silver_core_refresh",
        "description": "Refresh Silver internal, external, relationship, and domain tables.",
        "notebook_keys": [
            "07_silver_m5_calendar",
            "08_silver_m5_sell_prices",
            "09_silver_m5_sales_daily",
            "10_silver_retail_inventory",
            "11_silver_dataco_orders_shipments",
            "12_silver_synthetic_supplier_tables",
            "13_silver_external_risk_events_evidence",
            "14_silver_relationship_validation",
            "15_silver_domain_views"
        ],
        "depends_on": ["bronze_external_validation"],
        "layer": "silver",
        "critical": True
    },
    {
        "task_key": "gold_marts_refresh",
        "description": "Refresh Gold dimensions, dependency marts, risk marts, and dashboard tables.",
        "notebook_keys": [
            "16_gold_dim_suppliers",
            "17_gold_dim_products_skus",
            "18_gold_supplier_sku_dependency_mart",
            "19_gold_inventory_stockout_feature_mart",
            "20_gold_supplier_performance_mart",
            "21_gold_external_risk_event_mart",
            "22_gold_supplier_risk_scores",
            "23_gold_sku_stockout_risk_scores",
            "24_27_gold_rag_alerts_dashboards"
        ],
        "depends_on": ["silver_core_refresh"],
        "layer": "gold",
        "critical": True
    },
    {
        "task_key": "rag_context_refresh",
        "description": "Refresh RAG embeddings, retrieval index, and supplier chat context snapshots.",
        "notebook_keys": [
            "25_gold_rag_embeddings",
            "25b_gold_chat_context_snapshots",
            "26_rag_retrieval_test"
        ],
        "depends_on": ["gold_marts_refresh"],
        "layer": "rag",
        "critical": True
    },
    {
        "task_key": "agent_runtime_smoke_test",
        "description": "Run supplier risk agent smoke test.",
        "notebook_key": "27_langgraph_supplier_risk_agent",
        "depends_on": ["rag_context_refresh"],
        "layer": "agent",
        "critical": True
    },
    {
        "task_key": "agent_monitoring_eval",
        "description": "Run MLflow monitoring and deterministic agent evaluation suite.",
        "notebook_key": "28_mlflow_agent_monitoring_evaluation",
        "depends_on": ["agent_runtime_smoke_test"],
        "layer": "monitoring",
        "critical": True
    },
    {
        "task_key": "serving_health_check",
        "description": "Run serving wrapper health checks and update deployment manifest.",
        "notebook_key": "29_agent_serving_deployment_wrapper",
        "depends_on": ["agent_monitoring_eval"],
        "layer": "serving",
        "critical": True
    },
    {
        "task_key": "workflow_summary",
        "description": "Record Lakeflow orchestration manifest and workflow deployment status.",
        "notebook_key": "30_lakeflow_jobs_orchestration_deployment",
        "depends_on": ["serving_health_check"],
        "layer": "orchestration",
        "critical": True
    }
]

print("\nLakeflow task graph:")
for task in LAKEFLOW_TASK_GRAPH:
    print(
        f"{task['task_key']} | layer={task['layer']} | "
        f"depends_on={task['depends_on']} | critical={task['critical']}"
    )

# ─────────────────────────────────────────────────────────────
# Jobs API readiness check
# ─────────────────────────────────────────────────────────────

def get_databricks_api_token():
    """
    In some Databricks notebook contexts, apiToken is not exposed.
    If unavailable, later cells can still generate the job JSON payload
    and you can create the job through UI or CLI.
    """
    try:
        ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
        return ctx.apiToken().get()
    except Exception:
        return None


def normalize_workspace_url(host_or_url):
    if host_or_url is None:
        return None

    value = str(host_or_url).strip()

    if value.startswith("https://"):
        return value.rstrip("/")

    return f"https://{value}".rstrip("/")


DATABRICKS_WORKSPACE_URL = normalize_workspace_url(WORKSPACE_HOST or WORKSPACE_API_URL)
DATABRICKS_API_TOKEN = get_databricks_api_token()

jobs_api_ready = bool(DATABRICKS_WORKSPACE_URL and DATABRICKS_API_TOKEN)

api_readiness = {
    "workspace_url_available": bool(DATABRICKS_WORKSPACE_URL),
    "api_token_available": bool(DATABRICKS_API_TOKEN),
    "jobs_api_ready": jobs_api_ready,
    "create_or_update_job_enabled": CREATE_OR_UPDATE_JOB
}

print("\nJobs API readiness:")
print(json.dumps(api_readiness, indent=2))

if not jobs_api_ready:
    print(
        "\nJobs API token is not available from this notebook context. "
        "That is okay for now. We will still generate the Lakeflow Jobs config. "
        "Later, we can use Databricks CLI, a PAT secret, or the Jobs UI to deploy it."
    )

# ─────────────────────────────────────────────────────────────
# Deployment metadata
# ─────────────────────────────────────────────────────────────

workflow_deployment_id = f"workflow_deploy_{datetime.now(timezone.utc).strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}"

workflow_metadata = {
    "workflow_deployment_id": workflow_deployment_id,
    "project_name": PROJECT_NAME,
    "workflow_name": WORKFLOW_NAME,
    "workflow_display_name": WORKFLOW_DISPLAY_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_timezone_id": JOB_TIMEZONE_ID,
    "job_quartz_cron": JOB_QUARTZ_CRON,
    "create_or_update_job": CREATE_OR_UPDATE_JOB,
    "jobs_api_ready": jobs_api_ready,
    "notebook_base_path": NOTEBOOK_BASE_PATH,
    "current_notebook_path": CURRENT_NOTEBOOK_PATH,
    "workspace_host": WORKSPACE_HOST,
    "workspace_id": WORKSPACE_ID,
    "current_user": CURRENT_USER,
    "python_version": platform.python_version(),
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

print("\nWorkflow deployment metadata:")
print(json.dumps(workflow_metadata, indent=2))

print("\nCell 1 complete.")
print("Next: Cell 2 will validate notebook paths, required tables, and job graph integrity.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Replacement Cell 2
# Validate patched notebook paths, product tables, and task graph integrity
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

import pandas as pd
from pyspark.sql import functions as F

assert "NOTEBOOK_PATHS" in globals(), "NOTEBOOK_PATHS missing. Rerun Cell 1."
assert "LAKEFLOW_TASK_GRAPH" in globals(), "LAKEFLOW_TASK_GRAPH missing. Rerun Cell 1."
assert "REQUIRED_PRODUCT_TABLES" in globals(), "REQUIRED_PRODUCT_TABLES missing. Rerun Cell 1."
assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."

validation_started_at = datetime.now(timezone.utc)

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def workspace_get_status(path: str) -> dict:
    if not DATABRICKS_WORKSPACE_URL or not DATABRICKS_API_TOKEN:
        return {
            "path": path,
            "exists": None,
            "object_type": None,
            "language": None,
            "error": "Workspace API not available."
        }

    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.0/workspace/get-status"
    headers = {"Authorization": f"Bearer {DATABRICKS_API_TOKEN}"}
    params = {"path": path}

    try:
        response = requests.get(url, headers=headers, params=params, timeout=30)

        if response.status_code == 200:
            payload = response.json()
            return {
                "path": path,
                "exists": True,
                "object_type": payload.get("object_type"),
                "language": payload.get("language"),
                "object_id": str(payload.get("object_id")) if payload.get("object_id") is not None else None,
                "error": None
            }

        return {
            "path": path,
            "exists": False,
            "object_type": None,
            "language": None,
            "object_id": None,
            "error": f"HTTP {response.status_code}: {response.text[:300]}"
        }

    except Exception as e:
        return {
            "path": path,
            "exists": False,
            "object_type": None,
            "language": None,
            "object_id": None,
            "error": str(e)
        }


def table_exists(table_name: str) -> bool:
    try:
        spark.table(table_name).limit(1).collect()
        return True
    except Exception:
        return False


def get_table_row_count(table_name: str):
    try:
        return spark.table(table_name).count()
    except Exception:
        return None


def get_table_columns(table_name: str):
    try:
        return spark.table(table_name).columns
    except Exception:
        return []


def get_best_timestamp_column(columns):
    timestamp_candidates = [
        "gold_created_at",
        "created_at",
        "updated_at",
        "snapshot_timestamp",
        "snapshot_date",
        "monitoring_logged_at",
        "finished_at",
        "health_finished_at",
        "run_finished_at",
        "saved_at"
    ]

    for col_name in timestamp_candidates:
        if col_name in columns:
            return col_name

    return None


def get_latest_timestamp(table_name: str, timestamp_column: str):
    try:
        row = (
            spark.table(table_name)
            .select(F.max(F.col(timestamp_column)).alias("latest_ts"))
            .collect()[0]
        )
        return row["latest_ts"]
    except Exception:
        return None


# ─────────────────────────────────────────────────────────────
# Collect notebook keys used by the actual task graph
# ─────────────────────────────────────────────────────────────

used_notebook_keys = []

for task in LAKEFLOW_TASK_GRAPH:
    if "notebook_key" in task:
        used_notebook_keys.append(task["notebook_key"])

    if "notebook_keys" in task:
        used_notebook_keys.extend(task["notebook_keys"])

used_notebook_keys = sorted(set(used_notebook_keys))

# ─────────────────────────────────────────────────────────────
# 1. Validate notebook paths used by task graph
# ─────────────────────────────────────────────────────────────

notebook_validation_rows = []

for notebook_key in used_notebook_keys:
    notebook_path = NOTEBOOK_PATHS.get(notebook_key)

    if notebook_path is None:
        notebook_validation_rows.append({
            "validation_type": "notebook_path",
            "asset_key": notebook_key,
            "asset_name": None,
            "exists": False,
            "status": "FAIL",
            "object_type": None,
            "language": None,
            "object_id": None,
            "row_count": None,
            "timestamp_column": None,
            "latest_timestamp": None,
            "error": "Notebook key missing from NOTEBOOK_PATHS."
        })
        continue

    status = workspace_get_status(notebook_path)

    notebook_validation_rows.append({
        "validation_type": "notebook_path",
        "asset_key": notebook_key,
        "asset_name": notebook_path,
        "exists": status.get("exists"),
        "status": "PASS" if status.get("exists") else "FAIL",
        "object_type": status.get("object_type"),
        "language": status.get("language"),
        "object_id": status.get("object_id"),
        "row_count": None,
        "timestamp_column": None,
        "latest_timestamp": None,
        "error": status.get("error")
    })

notebook_validation_pd = pd.DataFrame(notebook_validation_rows)

print("Notebook path validation:")
display(spark.createDataFrame(notebook_validation_pd))

missing_notebooks = notebook_validation_pd[notebook_validation_pd["status"] == "FAIL"]

if len(missing_notebooks) > 0:
    print("Missing notebook paths:")
    display(spark.createDataFrame(missing_notebooks))
else:
    print("All task-graph notebook paths exist.")

# ─────────────────────────────────────────────────────────────
# 2. Validate required product tables
# ─────────────────────────────────────────────────────────────

table_validation_rows = []

for table_name in REQUIRED_PRODUCT_TABLES:
    exists_flag = table_exists(table_name)
    row_count = get_table_row_count(table_name) if exists_flag else None
    columns = get_table_columns(table_name) if exists_flag else []
    timestamp_column = get_best_timestamp_column(columns)
    latest_timestamp = get_latest_timestamp(table_name, timestamp_column) if timestamp_column else None

    if not exists_flag:
        status = "FAIL"
        error = "Table does not exist or is not accessible."
    elif row_count is None or row_count == 0:
        status = "FAIL"
        error = "Table exists but has no rows."
    else:
        status = "PASS"
        error = None

    table_validation_rows.append({
        "validation_type": "required_table",
        "asset_key": table_name,
        "asset_name": table_name,
        "exists": bool(exists_flag),
        "status": status,
        "object_type": "DELTA_TABLE",
        "language": None,
        "object_id": None,
        "row_count": int(row_count) if row_count is not None else None,
        "timestamp_column": timestamp_column,
        "latest_timestamp": str(latest_timestamp) if latest_timestamp is not None else None,
        "error": error
    })

table_validation_pd = pd.DataFrame(table_validation_rows)

print("\nRequired product table validation:")
display(spark.createDataFrame(table_validation_pd))

failed_tables = table_validation_pd[table_validation_pd["status"] == "FAIL"]

if len(failed_tables) > 0:
    print("Failed required table checks:")
    display(spark.createDataFrame(failed_tables))
else:
    print("All required product tables exist and are non-empty.")

# ─────────────────────────────────────────────────────────────
# 3. Validate task graph integrity
# ─────────────────────────────────────────────────────────────

task_keys = [task["task_key"] for task in LAKEFLOW_TASK_GRAPH]
task_key_set = set(task_keys)

graph_validation_rows = []

duplicate_task_keys = sorted([key for key in task_key_set if task_keys.count(key) > 1])

graph_validation_rows.append({
    "validation_check": "unique_task_keys",
    "status": "PASS" if len(duplicate_task_keys) == 0 else "FAIL",
    "details": json.dumps({"duplicate_task_keys": duplicate_task_keys})
})

missing_dependency_refs = []

for task in LAKEFLOW_TASK_GRAPH:
    for dep in task.get("depends_on", []):
        if dep not in task_key_set:
            missing_dependency_refs.append({
                "task_key": task["task_key"],
                "missing_dependency": dep
            })

graph_validation_rows.append({
    "validation_check": "dependency_references_exist",
    "status": "PASS" if len(missing_dependency_refs) == 0 else "FAIL",
    "details": json.dumps({"missing_dependency_refs": missing_dependency_refs})
})

missing_notebook_refs = []

for task in LAKEFLOW_TASK_GRAPH:
    if "notebook_key" in task:
        if task["notebook_key"] not in NOTEBOOK_PATHS:
            missing_notebook_refs.append({
                "task_key": task["task_key"],
                "missing_notebook_key": task["notebook_key"]
            })

    if "notebook_keys" in task:
        for notebook_key in task["notebook_keys"]:
            if notebook_key not in NOTEBOOK_PATHS:
                missing_notebook_refs.append({
                    "task_key": task["task_key"],
                    "missing_notebook_key": notebook_key
                })

graph_validation_rows.append({
    "validation_check": "notebook_references_exist",
    "status": "PASS" if len(missing_notebook_refs) == 0 else "FAIL",
    "details": json.dumps({"missing_notebook_refs": missing_notebook_refs})
})

def has_cycle(tasks):
    graph = {task["task_key"]: task.get("depends_on", []) for task in tasks}
    visiting = set()
    visited = set()

    def visit(node):
        if node in visiting:
            return True

        if node in visited:
            return False

        visiting.add(node)

        for parent in graph.get(node, []):
            if visit(parent):
                return True

        visiting.remove(node)
        visited.add(node)
        return False

    for node in graph:
        if visit(node):
            return True

    return False

cycle_found = has_cycle(LAKEFLOW_TASK_GRAPH)

graph_validation_rows.append({
    "validation_check": "no_cycles",
    "status": "PASS" if not cycle_found else "FAIL",
    "details": json.dumps({"cycle_found": cycle_found})
})

expected_layer_order = [
    "bronze",
    "silver",
    "gold",
    "rag",
    "agent",
    "monitoring",
    "serving",
    "orchestration"
]

task_layers = [task.get("layer") for task in LAKEFLOW_TASK_GRAPH]
unknown_layers = sorted([layer for layer in set(task_layers) if layer not in expected_layer_order])

graph_validation_rows.append({
    "validation_check": "known_layers",
    "status": "PASS" if len(unknown_layers) == 0 else "FAIL",
    "details": json.dumps({
        "expected_layer_order": expected_layer_order,
        "unknown_layers": unknown_layers
    })
})

graph_validation_pd = pd.DataFrame(graph_validation_rows)

print("\nTask graph validation:")
display(spark.createDataFrame(graph_validation_pd))

failed_graph_checks = graph_validation_pd[graph_validation_pd["status"] == "FAIL"]

# ─────────────────────────────────────────────────────────────
# 4. Overall validation result
# ─────────────────────────────────────────────────────────────

validation_finished_at = datetime.now(timezone.utc)

notebook_pass_count = int((notebook_validation_pd["status"] == "PASS").sum())
notebook_fail_count = int((notebook_validation_pd["status"] == "FAIL").sum())

table_pass_count = int((table_validation_pd["status"] == "PASS").sum())
table_fail_count = int((table_validation_pd["status"] == "FAIL").sum())

graph_pass_count = int((graph_validation_pd["status"] == "PASS").sum())
graph_fail_count = int((graph_validation_pd["status"] == "FAIL").sum())

overall_validation_passed = (
    notebook_fail_count == 0
    and table_fail_count == 0
    and graph_fail_count == 0
)

validation_summary = {
    "workflow_deployment_id": workflow_deployment_id,
    "workflow_name": WORKFLOW_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "notebook_pass_count": notebook_pass_count,
    "notebook_fail_count": notebook_fail_count,
    "table_pass_count": table_pass_count,
    "table_fail_count": table_fail_count,
    "graph_pass_count": graph_pass_count,
    "graph_fail_count": graph_fail_count,
    "overall_validation_passed": overall_validation_passed,
    "validation_duration_seconds": (validation_finished_at - validation_started_at).total_seconds(),
    "validation_started_at": validation_started_at.isoformat(),
    "validation_finished_at": validation_finished_at.isoformat()
}

print("\nWorkflow validation summary:")
print(json.dumps(validation_summary, indent=2))

if not overall_validation_passed:
    print(
        "\nValidation did not fully pass. Review failed notebook/table/graph checks above before creating the Lakeflow Job."
    )
else:
    print("\nAll workflow deployment validations passed.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 2A
# Discover actual notebook paths in workspace folder
# ─────────────────────────────────────────────────────────────

import json
import re
import requests
import pandas as pd
from pyspark.sql import functions as F

assert "NOTEBOOK_BASE_PATH" in globals(), "NOTEBOOK_BASE_PATH missing. Rerun Cell 1."
assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."
assert "NOTEBOOK_PATHS" in globals(), "NOTEBOOK_PATHS missing. Rerun Cell 1."

headers = {"Authorization": f"Bearer {DATABRICKS_API_TOKEN}"}

def workspace_list(path: str) -> list:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.0/workspace/list"
    params = {"path": path}

    response = requests.get(url, headers=headers, params=params, timeout=30)

    if response.status_code != 200:
        raise ValueError(f"Workspace list failed for {path}: HTTP {response.status_code} {response.text[:500]}")

    return response.json().get("objects", [])


def normalize_name(value: str) -> str:
    """
    Normalize names for fuzzy matching:
    - lowercase
    - remove file-copy suffixes like (1), (3)
    - remove spaces, underscores, dashes, punctuation
    """
    if value is None:
        return ""

    value = str(value).lower()
    value = re.sub(r"\(\d+\)", "", value)
    value = re.sub(r"[^a-z0-9]", "", value)

    return value


# List objects directly under notebook base path
workspace_objects = workspace_list(NOTEBOOK_BASE_PATH)

workspace_rows = []

for obj in workspace_objects:
    path = obj.get("path")
    name = path.split("/")[-1] if path else None

    workspace_rows.append({
        "object_name": name,
        "path": path,
        "object_type": obj.get("object_type"),
        "language": obj.get("language"),
        "object_id": str(obj.get("object_id")) if obj.get("object_id") is not None else None,
        "normalized_name": normalize_name(name)
    })

workspace_objects_pd = pd.DataFrame(workspace_rows).sort_values("object_name")

print(f"Objects found under: {NOTEBOOK_BASE_PATH}")
display(spark.createDataFrame(workspace_objects_pd))

# Build matching candidates for registered notebook keys
actual_notebooks_pd = workspace_objects_pd[
    workspace_objects_pd["object_type"].isin(["NOTEBOOK"])
].copy()

registered_rows = []

for notebook_key, registered_path in NOTEBOOK_PATHS.items():
    registered_name = registered_path.split("/")[-1]
    registered_norm = normalize_name(registered_name)

    exact_match = actual_notebooks_pd[
        actual_notebooks_pd["path"] == registered_path
    ]

    normalized_match = actual_notebooks_pd[
        actual_notebooks_pd["normalized_name"] == registered_norm
    ]

    prefix_match = actual_notebooks_pd[
        actual_notebooks_pd["normalized_name"].str.startswith(registered_norm, na=False)
        | actual_notebooks_pd["normalized_name"].apply(lambda x: registered_norm.startswith(x) if isinstance(x, str) else False)
    ]

    if len(exact_match) > 0:
        match_status = "EXACT_PATH_MATCH"
        matched_path = exact_match.iloc[0]["path"]
        matched_name = exact_match.iloc[0]["object_name"]
    elif len(normalized_match) > 0:
        match_status = "NORMALIZED_NAME_MATCH"
        matched_path = normalized_match.iloc[0]["path"]
        matched_name = normalized_match.iloc[0]["object_name"]
    elif len(prefix_match) > 0:
        match_status = "POSSIBLE_PREFIX_MATCH"
        matched_path = prefix_match.iloc[0]["path"]
        matched_name = prefix_match.iloc[0]["object_name"]
    else:
        match_status = "NO_MATCH"
        matched_path = None
        matched_name = None

    registered_rows.append({
        "notebook_key": notebook_key,
        "registered_path": registered_path,
        "registered_name": registered_name,
        "match_status": match_status,
        "matched_name": matched_name,
        "matched_path": matched_path
    })

notebook_path_match_pd = pd.DataFrame(registered_rows)

print("\nNotebook registry match results:")
display(spark.createDataFrame(notebook_path_match_pd))

print("\nUnmatched registered notebooks:")
unmatched_pd = notebook_path_match_pd[notebook_path_match_pd["match_status"] == "NO_MATCH"]

if len(unmatched_pd) > 0:
    display(spark.createDataFrame(unmatched_pd))
else:
    print("All registered notebooks matched.")

# Patch NOTEBOOK_PATHS where we have exact, normalized, or prefix matches.
patched_count = 0

for row in notebook_path_match_pd.to_dict(orient="records"):
    if row["match_status"] in ["EXACT_PATH_MATCH", "NORMALIZED_NAME_MATCH", "POSSIBLE_PREFIX_MATCH"]:
        if row["matched_path"] and NOTEBOOK_PATHS[row["notebook_key"]] != row["matched_path"]:
            NOTEBOOK_PATHS[row["notebook_key"]] = row["matched_path"]
            patched_count += 1

print(f"\nPatched NOTEBOOK_PATHS entries: {patched_count}")

print("\nCurrent patched notebook paths:")
for key, path in NOTEBOOK_PATHS.items():
    print(f"{key}: {path}")

print("\nCell 2A complete.")
print("Next: rerun Replacement Cell 2 after reviewing unmatched notebooks.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 2B
# Patch notebook registry based on workspace discovery
# ─────────────────────────────────────────────────────────────

assert "NOTEBOOK_BASE_PATH" in globals(), "NOTEBOOK_BASE_PATH missing. Rerun Cell 1."
assert "NOTEBOOK_PATHS" in globals(), "NOTEBOOK_PATHS missing. Rerun Cell 1."
assert "LAKEFLOW_TASK_GRAPH" in globals(), "LAKEFLOW_TASK_GRAPH missing. Rerun Cell 1."

# Manual corrections from Cell 2A workspace discovery
NOTEBOOK_PATHS["12_silver_synthetic_supplier_tables"] = (
    f"{NOTEBOOK_BASE_PATH}/12_silver_synthetic_internal_tables"
)

NOTEBOOK_PATHS["13_silver_external_risk_events_evidence"] = (
    f"{NOTEBOOK_BASE_PATH}/13_silver_external_risk_events"
)

NOTEBOOK_PATHS["26_rag_retrieval_test"] = (
    f"{NOTEBOOK_BASE_PATH}/Testing RAG"
)

# Auto-patched copy-suffix notebook paths from Cell 2A discovery
NOTEBOOK_PATHS["16_gold_dim_suppliers"] = (
    f"{NOTEBOOK_BASE_PATH}/16_gold_dim_suppliers (1)"
)

NOTEBOOK_PATHS["17_gold_dim_products_skus"] = (
    f"{NOTEBOOK_BASE_PATH}/17_gold_dim_products_skus (1)"
)

NOTEBOOK_PATHS["19_gold_inventory_stockout_feature_mart"] = (
    f"{NOTEBOOK_BASE_PATH}/19_gold_inventory_stockout_feature_mart (1)"
)

NOTEBOOK_PATHS["20_gold_supplier_performance_mart"] = (
    f"{NOTEBOOK_BASE_PATH}/20_gold_supplier_performance_mart (1)"
)

NOTEBOOK_PATHS["21_gold_external_risk_event_mart"] = (
    f"{NOTEBOOK_BASE_PATH}/21_gold_external_risk_event_mart (3)"
)

NOTEBOOK_PATHS["23_gold_sku_stockout_risk_scores"] = (
    f"{NOTEBOOK_BASE_PATH}/23_gold_sku_stockout_risk_scores (1)"
)

NOTEBOOK_PATHS["24_27_gold_rag_alerts_dashboards"] = (
    f"{NOTEBOOK_BASE_PATH}/24_27_gold_rag_alerts_dashboards (1)"
)

NOTEBOOK_PATHS["25_gold_rag_embeddings"] = (
    f"{NOTEBOOK_BASE_PATH}/25_gold_rag_embeddings (3)"
)

# 15_silver_domain_views does not exist in the workspace folder.
# Remove it from the Silver task so the Lakeflow Job does not reference a missing notebook.
for task in LAKEFLOW_TASK_GRAPH:
    if task["task_key"] == "silver_core_refresh":
        original_notebook_keys = task["notebook_keys"]

        task["notebook_keys"] = [
            notebook_key
            for notebook_key in original_notebook_keys
            if notebook_key != "15_silver_domain_views"
        ]

        print("Updated silver_core_refresh notebook list:")
        for notebook_key in task["notebook_keys"]:
            print("-", notebook_key)

print("\nPatched NOTEBOOK_PATHS:")
for key, value in NOTEBOOK_PATHS.items():
    print(f"{key}: {value}")

print("\nCell 2B complete.")
print("Next: rerun Replacement Cell 2 to validate the patched registry.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 3
# Build Lakeflow Jobs API payload
# ─────────────────────────────────────────────────────────────

import json
from datetime import datetime, timezone

assert "overall_validation_passed" in globals(), "overall_validation_passed missing. Rerun Cell 2."
assert overall_validation_passed is True, "Workflow validation failed. Fix Cell 2 checks before building job payload."

assert "NOTEBOOK_PATHS" in globals(), "NOTEBOOK_PATHS missing. Rerun Cell 1."
assert "LAKEFLOW_TASK_GRAPH" in globals(), "LAKEFLOW_TASK_GRAPH missing. Rerun Cell 1."
assert "JOB_BASE_PARAMETERS" in globals(), "JOB_BASE_PARAMETERS missing. Rerun Cell 1."

# ─────────────────────────────────────────────────────────────
# Cluster / compute config
# ─────────────────────────────────────────────────────────────
# For product-readiness, we use job clusters rather than all-purpose interactive clusters.
# This config is intentionally modest. You can tune node_type_id / autoscale later.

JOB_CLUSTER_KEY = "supplysage_job_cluster"

JOB_CLUSTER_SPEC = {
    "job_cluster_key": JOB_CLUSTER_KEY,
    "new_cluster": {
        "spark_version": "16.4.x-scala2.12",
        "node_type_id": "i3.xlarge",
        "driver_node_type_id": "i3.xlarge",
        "autoscale": {
            "min_workers": 1,
            "max_workers": 3
        },
        "spark_conf": {
            "spark.databricks.delta.schema.autoMerge.enabled": "true"
        },
        "custom_tags": {
            "project": "supplysage_ai",
            "environment": "production",
            "owner": CREATED_BY
        }
    }
}

# ─────────────────────────────────────────────────────────────
# Expand grouped logical tasks into executable notebook tasks
# ─────────────────────────────────────────────────────────────
# Databricks Jobs task dependencies reference concrete task keys.
# For grouped layers such as silver_core_refresh, we create one task per notebook
# and chain them in order. Then downstream logical tasks depend on the final task
# in that group.

expanded_tasks = []
logical_task_to_terminal_task = {}
logical_task_to_first_task = {}

for logical_task in LAKEFLOW_TASK_GRAPH:
    logical_key = logical_task["task_key"]

    if "notebook_key" in logical_task:
        executable_task_key = logical_key

        depends_on_terminal_tasks = [
            logical_task_to_terminal_task.get(dep, dep)
            for dep in logical_task.get("depends_on", [])
        ]

        expanded_tasks.append({
            "task_key": executable_task_key,
            "logical_task_key": logical_key,
            "description": logical_task.get("description"),
            "notebook_key": logical_task["notebook_key"],
            "depends_on": depends_on_terminal_tasks,
            "layer": logical_task.get("layer"),
            "critical": logical_task.get("critical", True)
        })

        logical_task_to_first_task[logical_key] = executable_task_key
        logical_task_to_terminal_task[logical_key] = executable_task_key

    elif "notebook_keys" in logical_task:
        previous_task_key = None
        first_task_key = None

        for idx, notebook_key in enumerate(logical_task["notebook_keys"], start=1):
            executable_task_key = f"{logical_key}_{idx:02d}_{notebook_key}"

            if idx == 1:
                depends_on_terminal_tasks = [
                    logical_task_to_terminal_task.get(dep, dep)
                    for dep in logical_task.get("depends_on", [])
                ]
                first_task_key = executable_task_key
            else:
                depends_on_terminal_tasks = [previous_task_key]

            expanded_tasks.append({
                "task_key": executable_task_key,
                "logical_task_key": logical_key,
                "description": f"{logical_task.get('description')} Step {idx}: {notebook_key}",
                "notebook_key": notebook_key,
                "depends_on": depends_on_terminal_tasks,
                "layer": logical_task.get("layer"),
                "critical": logical_task.get("critical", True)
            })

            previous_task_key = executable_task_key

        logical_task_to_first_task[logical_key] = first_task_key
        logical_task_to_terminal_task[logical_key] = previous_task_key

    else:
        raise ValueError(f"Task {logical_key} has no notebook_key or notebook_keys.")

print("Expanded executable tasks:")
for task in expanded_tasks:
    print(
        f"{task['task_key']} | notebook={task['notebook_key']} | "
        f"depends_on={task['depends_on']}"
    )

# ─────────────────────────────────────────────────────────────
# Build Databricks Jobs task definitions
# ─────────────────────────────────────────────────────────────

def build_task_parameters(task: dict) -> dict:
    params = dict(JOB_BASE_PARAMETERS)

    params.update({
        "workflow_deployment_id": workflow_deployment_id,
        "task_key": task["task_key"],
        "logical_task_key": task["logical_task_key"],
        "layer": task["layer"],
        "triggered_by": "lakeflow_job"
    })

    return params


job_tasks = []

for task in expanded_tasks:
    notebook_key = task["notebook_key"]
    notebook_path = NOTEBOOK_PATHS[notebook_key]

    job_task = {
        "task_key": task["task_key"],
        "description": task["description"],
        "depends_on": [
            {"task_key": dep_task_key}
            for dep_task_key in task.get("depends_on", [])
        ],
        "notebook_task": {
            "notebook_path": notebook_path,
            "base_parameters": build_task_parameters(task)
        },
        "job_cluster_key": JOB_CLUSTER_KEY,
        "max_retries": DEFAULT_MAX_RETRIES,
        "min_retry_interval_millis": DEFAULT_MIN_RETRY_INTERVAL_MILLIS,
        "timeout_seconds": DEFAULT_TIMEOUT_SECONDS
    }

    # Databricks API allows depends_on to be absent for root tasks.
    if len(job_task["depends_on"]) == 0:
        job_task.pop("depends_on")

    job_tasks.append(job_task)

# ─────────────────────────────────────────────────────────────
# Job payload
# ─────────────────────────────────────────────────────────────

lakeflow_job_payload = {
    "name": WORKFLOW_DISPLAY_NAME,
    "tags": {
        "project": "supplysage_ai",
        "workflow_name": WORKFLOW_NAME,
        "workflow_version": WORKFLOW_VERSION,
        "environment": "production",
        "owner": CREATED_BY
    },
    "schedule": {
        "quartz_cron_expression": JOB_QUARTZ_CRON,
        "timezone_id": JOB_TIMEZONE_ID,
        "pause_status": "PAUSED"
    },
    "max_concurrent_runs": 1,
    "job_clusters": [
        JOB_CLUSTER_SPEC
    ],
    "tasks": job_tasks,
    "format": "MULTI_TASK"
}

# Note:
# We set pause_status=PAUSED by default so creating the job will not immediately schedule runs.
# You can unpause from the Jobs UI after one successful manual run.

job_payload_summary = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_name": lakeflow_job_payload["name"],
    "task_count": len(job_tasks),
    "job_cluster_key": JOB_CLUSTER_KEY,
    "schedule": lakeflow_job_payload["schedule"],
    "max_concurrent_runs": lakeflow_job_payload["max_concurrent_runs"],
    "pause_status": lakeflow_job_payload["schedule"]["pause_status"],
    "created_at_utc": datetime.now(timezone.utc).isoformat()
}

print("\nLakeflow Jobs payload summary:")
print(json.dumps(job_payload_summary, indent=2))

print("\nLakeflow Jobs API payload:")
print(json.dumps(lakeflow_job_payload, indent=2)[:6000])
print("\n...payload preview truncated for display...")

print("\nCell 3 complete.")
print("Next: Cell 4 will save the Lakeflow job manifest and full JSON payload to Delta.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 4
# Save Lakeflow Jobs manifest + full API payload to Delta
# ─────────────────────────────────────────────────────────────

import json
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    TimestampType
)

assert "lakeflow_job_payload" in globals(), "lakeflow_job_payload missing. Rerun Cell 3."
assert "expanded_tasks" in globals(), "expanded_tasks missing. Rerun Cell 3."
assert "job_tasks" in globals(), "job_tasks missing. Rerun Cell 3."
assert "workflow_deployment_id" in globals(), "workflow_deployment_id missing. Rerun Cell 1."

manifest_created_at = datetime.now(timezone.utc)

# ─────────────────────────────────────────────────────────────
# Job-level manifest row
# ─────────────────────────────────────────────────────────────

job_manifest_row = [{
    "workflow_deployment_id": workflow_deployment_id,
    "project_name": PROJECT_NAME,
    "workflow_name": WORKFLOW_NAME,
    "workflow_display_name": WORKFLOW_DISPLAY_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_name": lakeflow_job_payload.get("name"),
    "job_id": None,
    "job_created_or_updated": False,
    "create_or_update_job_enabled": bool(CREATE_OR_UPDATE_JOB),
    "jobs_api_ready": bool(jobs_api_ready),
    "task_count": int(len(job_tasks)),
    "schedule_quartz_cron": JOB_QUARTZ_CRON,
    "schedule_timezone_id": JOB_TIMEZONE_ID,
    "schedule_pause_status": lakeflow_job_payload.get("schedule", {}).get("pause_status"),
    "max_concurrent_runs": int(lakeflow_job_payload.get("max_concurrent_runs", 1)),
    "job_cluster_key": JOB_CLUSTER_KEY,
    "notebook_base_path": NOTEBOOK_BASE_PATH,
    "workspace_host": DATABRICKS_WORKSPACE_URL,
    "workspace_id": str(WORKSPACE_ID),
    "current_user": CURRENT_USER,
    "validation_passed": bool(overall_validation_passed),
    "deployment_status": "payload_generated_not_created",
    "lakeflow_job_payload_json": json.dumps(lakeflow_job_payload, indent=2, default=str),
    "job_payload_summary_json": json.dumps(job_payload_summary, indent=2, default=str),
    "created_at": manifest_created_at
}]

job_manifest_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("project_name", StringType(), True),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_display_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("job_name", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("job_created_or_updated", BooleanType(), True),
    StructField("create_or_update_job_enabled", BooleanType(), True),
    StructField("jobs_api_ready", BooleanType(), True),
    StructField("task_count", IntegerType(), True),
    StructField("schedule_quartz_cron", StringType(), True),
    StructField("schedule_timezone_id", StringType(), True),
    StructField("schedule_pause_status", StringType(), True),
    StructField("max_concurrent_runs", IntegerType(), True),
    StructField("job_cluster_key", StringType(), True),
    StructField("notebook_base_path", StringType(), True),
    StructField("workspace_host", StringType(), True),
    StructField("workspace_id", StringType(), True),
    StructField("current_user", StringType(), True),
    StructField("validation_passed", BooleanType(), True),
    StructField("deployment_status", StringType(), True),
    StructField("lakeflow_job_payload_json", StringType(), True),
    StructField("job_payload_summary_json", StringType(), True),
    StructField("created_at", TimestampType(), True),
])

job_manifest_df = spark.createDataFrame(job_manifest_row, schema=job_manifest_schema)

(
    job_manifest_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(WORKFLOW_JOB_MANIFEST_TABLE)
)

print(f"Saved workflow job manifest to: {WORKFLOW_JOB_MANIFEST_TABLE}")

# ─────────────────────────────────────────────────────────────
# Task-level health / manifest rows
# ─────────────────────────────────────────────────────────────

task_manifest_rows = []

for idx, task in enumerate(expanded_tasks, start=1):
    task_manifest_rows.append({
        "workflow_deployment_id": workflow_deployment_id,
        "workflow_name": WORKFLOW_NAME,
        "workflow_version": WORKFLOW_VERSION,
        "task_order": int(idx),
        "task_key": task["task_key"],
        "logical_task_key": task["logical_task_key"],
        "layer": task["layer"],
        "critical": bool(task["critical"]),
        "description": task["description"],
        "notebook_key": task["notebook_key"],
        "notebook_path": NOTEBOOK_PATHS[task["notebook_key"]],
        "depends_on_json": json.dumps(task.get("depends_on", []), default=str),
        "job_cluster_key": JOB_CLUSTER_KEY,
        "max_retries": int(DEFAULT_MAX_RETRIES),
        "timeout_seconds": int(DEFAULT_TIMEOUT_SECONDS),
        "validation_status": "PASS",
        "created_at": manifest_created_at
    })

task_manifest_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("task_order", IntegerType(), True),
    StructField("task_key", StringType(), True),
    StructField("logical_task_key", StringType(), True),
    StructField("layer", StringType(), True),
    StructField("critical", BooleanType(), True),
    StructField("description", StringType(), True),
    StructField("notebook_key", StringType(), True),
    StructField("notebook_path", StringType(), True),
    StructField("depends_on_json", StringType(), True),
    StructField("job_cluster_key", StringType(), True),
    StructField("max_retries", IntegerType(), True),
    StructField("timeout_seconds", IntegerType(), True),
    StructField("validation_status", StringType(), True),
    StructField("created_at", TimestampType(), True),
])

task_manifest_df = spark.createDataFrame(task_manifest_rows, schema=task_manifest_schema)

(
    task_manifest_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(WORKFLOW_TASK_HEALTH_TABLE)
)

print(f"Saved workflow task manifest to: {WORKFLOW_TASK_HEALTH_TABLE}")

# ─────────────────────────────────────────────────────────────
# Preview
# ─────────────────────────────────────────────────────────────

print("\nManifest saved for workflow_deployment_id:", workflow_deployment_id)
print("Deployment status: payload_generated_not_created")

display(
    spark.table(WORKFLOW_JOB_MANIFEST_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_name",
        "task_count",
        "schedule_quartz_cron",
        "schedule_timezone_id",
        "schedule_pause_status",
        "jobs_api_ready",
        "create_or_update_job_enabled",
        "job_created_or_updated",
        "deployment_status",
        "created_at"
    )
)

display(
    spark.table(WORKFLOW_TASK_HEALTH_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "task_order",
        "task_key",
        "logical_task_key",
        "layer",
        "notebook_key",
        "critical",
        "validation_status"
    )
    .orderBy("task_order")
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 5
# Dry-run Databricks Jobs deployer
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F

assert "lakeflow_job_payload" in globals(), "lakeflow_job_payload missing. Rerun Cell 3."
assert "jobs_api_ready" in globals(), "jobs_api_ready missing. Rerun Cell 1."
assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."

assert jobs_api_ready is True, "Jobs API is not ready. Cannot query Databricks Jobs API."

dry_run_started_at = datetime.now(timezone.utc)

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

def list_jobs_by_name(job_name: str) -> list:
    """
    List Databricks jobs and return jobs whose settings.name matches job_name.
    Uses Jobs API 2.1 list endpoint.
    """
    matching_jobs = []
    page_token = None

    while True:
        url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/list"

        params = {
            "limit": 25,
            "name": job_name
        }

        if page_token:
            params["page_token"] = page_token

        response = requests.get(
            url,
            headers=headers,
            params=params,
            timeout=30
        )

        if response.status_code != 200:
            raise ValueError(
                f"Jobs list failed: HTTP {response.status_code} {response.text[:500]}"
            )

        payload = response.json()

        for job in payload.get("jobs", []):
            settings = job.get("settings", {}) or {}

            if settings.get("name") == job_name:
                matching_jobs.append(job)

        page_token = payload.get("next_page_token")

        if not page_token:
            break

    return matching_jobs


target_job_name = lakeflow_job_payload["name"]
existing_jobs = list_jobs_by_name(target_job_name)

if len(existing_jobs) == 0:
    proposed_action = "CREATE_JOB"
    existing_job_id = None
elif len(existing_jobs) == 1:
    proposed_action = "RESET_EXISTING_JOB"
    existing_job_id = str(existing_jobs[0].get("job_id"))
else:
    proposed_action = "AMBIGUOUS_MULTIPLE_JOBS_FOUND"
    existing_job_id = ",".join([str(job.get("job_id")) for job in existing_jobs])

dry_run_finished_at = datetime.now(timezone.utc)

dry_run_result = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_name": target_job_name,
    "existing_job_count": len(existing_jobs),
    "existing_job_id": existing_job_id,
    "proposed_action": proposed_action,
    "create_or_update_job_enabled": bool(CREATE_OR_UPDATE_JOB),
    "jobs_api_ready": bool(jobs_api_ready),
    "would_create_or_update": bool(CREATE_OR_UPDATE_JOB and proposed_action in ["CREATE_JOB", "RESET_EXISTING_JOB"]),
    "dry_run_started_at": dry_run_started_at.isoformat(),
    "dry_run_finished_at": dry_run_finished_at.isoformat(),
    "dry_run_duration_seconds": (dry_run_finished_at - dry_run_started_at).total_seconds()
}

print("Lakeflow Jobs dry-run result:")
print(json.dumps(dry_run_result, indent=2))

if proposed_action == "AMBIGUOUS_MULTIPLE_JOBS_FOUND":
    print(
        "\nMultiple jobs with the same name were found. Do not auto-deploy until duplicates are resolved."
    )
elif CREATE_OR_UPDATE_JOB:
    print("\nCREATE_OR_UPDATE_JOB is True. The next deploy cell can create/update the Databricks job.")
else:
    print(
        "\nCREATE_OR_UPDATE_JOB is False. This was a safe dry run only. "
        "No Databricks job was created or updated."
    )

# Persist dry-run result by updating the latest manifest row status.
dry_run_update_df = spark.createDataFrame([{
    "workflow_deployment_id": workflow_deployment_id,
    "job_name": target_job_name,
    "existing_job_count": int(len(existing_jobs)),
    "existing_job_id": existing_job_id,
    "proposed_action": proposed_action,
    "create_or_update_job_enabled": bool(CREATE_OR_UPDATE_JOB),
    "would_create_or_update": bool(CREATE_OR_UPDATE_JOB and proposed_action in ["CREATE_JOB", "RESET_EXISTING_JOB"]),
    "dry_run_result_json": json.dumps(dry_run_result, indent=2, default=str),
    "dry_run_at": dry_run_finished_at
}])

DRY_RUN_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_deploy_dry_run"

(
    dry_run_update_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(DRY_RUN_TABLE)
)

print(f"\nSaved dry-run deploy result to: {DRY_RUN_TABLE}")

display(
    spark.table(DRY_RUN_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_name",
        "existing_job_count",
        "existing_job_id",
        "proposed_action",
        "create_or_update_job_enabled",
        "would_create_or_update",
        "dry_run_at"
    )
    .orderBy(F.desc("dry_run_at"))
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 5A
# Save dry-run deploy result with explicit schema
# Fixes: [CANNOT_DETERMINE_TYPE] from null existing_job_id
# ─────────────────────────────────────────────────────────────

import json
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    TimestampType
)

assert "dry_run_result" in globals(), "dry_run_result missing. Rerun Cell 5."
assert "workflow_deployment_id" in globals(), "workflow_deployment_id missing. Rerun Cell 1."

DRY_RUN_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_deploy_dry_run"

dry_run_at = datetime.now(timezone.utc)

dry_run_rows = [{
    "workflow_deployment_id": str(dry_run_result.get("workflow_deployment_id")),
    "job_name": str(dry_run_result.get("job_name")),
    "existing_job_count": int(dry_run_result.get("existing_job_count") or 0),
    "existing_job_id": (
        str(dry_run_result.get("existing_job_id"))
        if dry_run_result.get("existing_job_id") is not None
        else None
    ),
    "proposed_action": str(dry_run_result.get("proposed_action")),
    "create_or_update_job_enabled": bool(dry_run_result.get("create_or_update_job_enabled")),
    "jobs_api_ready": bool(dry_run_result.get("jobs_api_ready")),
    "would_create_or_update": bool(dry_run_result.get("would_create_or_update")),
    "dry_run_result_json": json.dumps(dry_run_result, indent=2, default=str),
    "dry_run_at": dry_run_at
}]

dry_run_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("job_name", StringType(), True),
    StructField("existing_job_count", IntegerType(), True),
    StructField("existing_job_id", StringType(), True),
    StructField("proposed_action", StringType(), True),
    StructField("create_or_update_job_enabled", BooleanType(), True),
    StructField("jobs_api_ready", BooleanType(), True),
    StructField("would_create_or_update", BooleanType(), True),
    StructField("dry_run_result_json", StringType(), True),
    StructField("dry_run_at", TimestampType(), True),
])

dry_run_update_df = spark.createDataFrame(dry_run_rows, schema=dry_run_schema)

(
    dry_run_update_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(DRY_RUN_TABLE)
)

print(f"Saved dry-run deploy result to: {DRY_RUN_TABLE}")

display(
    spark.table(DRY_RUN_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_name",
        "existing_job_count",
        "existing_job_id",
        "proposed_action",
        "create_or_update_job_enabled",
        "jobs_api_ready",
        "would_create_or_update",
        "dry_run_at"
    )
    .orderBy(F.desc("dry_run_at"))
)

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 6
# Create or update the actual Databricks Lakeflow Job
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    TimestampType
)

assert "lakeflow_job_payload" in globals(), "lakeflow_job_payload missing. Rerun Cell 3."
assert "dry_run_result" in globals(), "dry_run_result missing. Rerun Cell 5."
assert "jobs_api_ready" in globals(), "jobs_api_ready missing. Rerun Cell 1."
assert jobs_api_ready is True, "Jobs API is not ready."

# This is the real deployment switch for this cell.
DEPLOY_JOB_NOW = True

deploy_started_at = datetime.now(timezone.utc)

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

target_job_name = lakeflow_job_payload["name"]
proposed_action = dry_run_result.get("proposed_action")
existing_job_id = dry_run_result.get("existing_job_id")

if proposed_action not in ["CREATE_JOB", "RESET_EXISTING_JOB"]:
    raise ValueError(
        f"Unsafe deploy action: {proposed_action}. "
        "Expected CREATE_JOB or RESET_EXISTING_JOB."
    )

if not DEPLOY_JOB_NOW:
    raise ValueError("DEPLOY_JOB_NOW is False. Set to True only when ready to create/update the job.")

# ─────────────────────────────────────────────────────────────
# Deploy helpers
# ─────────────────────────────────────────────────────────────

def create_databricks_job(job_payload: dict) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/create"

    response = requests.post(
        url,
        headers=headers,
        data=json.dumps(job_payload),
        timeout=120
    )

    if response.status_code not in [200, 201]:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:2000],
            "job_id": None
        }

    payload = response.json()

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": json.dumps(payload, indent=2),
        "job_id": str(payload.get("job_id")) if payload.get("job_id") is not None else None
    }


def reset_databricks_job(job_id: str, job_payload: dict) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/reset"

    reset_payload = {
        "job_id": int(job_id),
        "new_settings": job_payload
    }

    response = requests.post(
        url,
        headers=headers,
        data=json.dumps(reset_payload),
        timeout=120
    )

    if response.status_code not in [200, 201]:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:2000],
            "job_id": str(job_id)
        }

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": response.text[:2000],
        "job_id": str(job_id)
    }


# ─────────────────────────────────────────────────────────────
# Execute deployment
# ─────────────────────────────────────────────────────────────

if proposed_action == "CREATE_JOB":
    deploy_api_result = create_databricks_job(lakeflow_job_payload)
    actual_action = "created"

elif proposed_action == "RESET_EXISTING_JOB":
    deploy_api_result = reset_databricks_job(existing_job_id, lakeflow_job_payload)
    actual_action = "reset"

else:
    raise ValueError(f"Unsupported proposed_action: {proposed_action}")

deploy_finished_at = datetime.now(timezone.utc)

job_id = deploy_api_result.get("job_id")
deployment_succeeded = bool(deploy_api_result.get("ok"))

if deployment_succeeded:
    deployment_status = "job_created_or_updated"
else:
    deployment_status = "job_deploy_failed"

deployment_result = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_name": target_job_name,
    "job_id": job_id,
    "proposed_action": proposed_action,
    "actual_action": actual_action,
    "deployment_succeeded": deployment_succeeded,
    "deployment_status": deployment_status,
    "status_code": deploy_api_result.get("status_code"),
    "api_response_preview": deploy_api_result.get("response_text"),
    "task_count": len(job_tasks),
    "schedule_pause_status": lakeflow_job_payload.get("schedule", {}).get("pause_status"),
    "deploy_started_at": deploy_started_at.isoformat(),
    "deploy_finished_at": deploy_finished_at.isoformat(),
    "deploy_duration_seconds": (deploy_finished_at - deploy_started_at).total_seconds()
}

print("Lakeflow Jobs deployment result:")
print(json.dumps(deployment_result, indent=2))

# ─────────────────────────────────────────────────────────────
# Save deployment event to Delta
# ─────────────────────────────────────────────────────────────

DEPLOYMENT_EVENTS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_deployment_events"

deployment_event_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("job_name", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("proposed_action", StringType(), True),
    StructField("actual_action", StringType(), True),
    StructField("deployment_succeeded", BooleanType(), True),
    StructField("deployment_status", StringType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("task_count", IntegerType(), True),
    StructField("schedule_pause_status", StringType(), True),
    StructField("api_response_preview", StringType(), True),
    StructField("deployment_result_json", StringType(), True),
    StructField("deploy_started_at", TimestampType(), True),
    StructField("deploy_finished_at", TimestampType(), True),
])

deployment_event_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "workflow_name": WORKFLOW_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_name": target_job_name,
    "job_id": job_id,
    "proposed_action": proposed_action,
    "actual_action": actual_action,
    "deployment_succeeded": deployment_succeeded,
    "deployment_status": deployment_status,
    "status_code": int(deploy_api_result.get("status_code")) if deploy_api_result.get("status_code") is not None else None,
    "task_count": int(len(job_tasks)),
    "schedule_pause_status": lakeflow_job_payload.get("schedule", {}).get("pause_status"),
    "api_response_preview": str(deploy_api_result.get("response_text"))[:2000],
    "deployment_result_json": json.dumps(deployment_result, indent=2, default=str),
    "deploy_started_at": deploy_started_at,
    "deploy_finished_at": deploy_finished_at
}]

deployment_event_df = spark.createDataFrame(
    deployment_event_rows,
    schema=deployment_event_schema
)

(
    deployment_event_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(DEPLOYMENT_EVENTS_TABLE)
)

print(f"\nSaved deployment event to: {DEPLOYMENT_EVENTS_TABLE}")

# ─────────────────────────────────────────────────────────────
# Update latest job manifest row
# ─────────────────────────────────────────────────────────────

deployment_update_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": job_id,
    "job_created_or_updated": deployment_succeeded,
    "deployment_status": deployment_status,
    "deployment_updated_at": deploy_finished_at
}]

deployment_update_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("job_id", StringType(), True),
    StructField("job_created_or_updated", BooleanType(), True),
    StructField("deployment_status", StringType(), True),
    StructField("deployment_updated_at", TimestampType(), True),
])

deployment_update_df = spark.createDataFrame(
    deployment_update_rows,
    schema=deployment_update_schema
)

deployment_update_df.createOrReplaceTempView("workflow_deployment_update")

spark.sql(f"""
MERGE INTO {WORKFLOW_JOB_MANIFEST_TABLE} AS target
USING workflow_deployment_update AS source
ON target.workflow_deployment_id = source.workflow_deployment_id
WHEN MATCHED THEN UPDATE SET
  target.job_id = source.job_id,
  target.job_created_or_updated = source.job_created_or_updated,
  target.deployment_status = source.deployment_status
""")

print(f"Updated workflow manifest table: {WORKFLOW_JOB_MANIFEST_TABLE}")

display(
    spark.table(WORKFLOW_JOB_MANIFEST_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_name",
        "job_id",
        "task_count",
        "schedule_pause_status",
        "job_created_or_updated",
        "deployment_status",
        "created_at"
    )
)

if deployment_succeeded:
    print("\nLakeflow Job created/updated successfully.")
    print(f"Job ID: {job_id}")
    print("The job is created in PAUSED state by design.")
else:
    print("\nLakeflow Job deployment failed.")
    print("Review api_response_preview above.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 7
# Verify deployed Lakeflow Job and save final deployment summary
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    BooleanType,
    TimestampType
)

assert "deployment_result" in globals(), "deployment_result missing. Rerun Cell 6A."
assert "serverless_job_payload" in globals(), "serverless_job_payload missing. Rerun Cell 6A."
assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."

verify_started_at = datetime.now(timezone.utc)

JOB_ID = str(deployment_result.get("job_id"))

if JOB_ID is None or JOB_ID == "None":
    raise ValueError("JOB_ID is missing. Job deployment did not succeed.")

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

# ─────────────────────────────────────────────────────────────
# Fetch deployed job settings from Databricks Jobs API
# ─────────────────────────────────────────────────────────────

def get_databricks_job(job_id: str) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/get"
    params = {"job_id": job_id}

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=60
    )

    if response.status_code != 200:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:3000],
            "job": None
        }

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": response.text[:3000],
        "job": response.json()
    }


job_get_result = get_databricks_job(JOB_ID)

if not job_get_result["ok"]:
    raise ValueError(
        f"Could not verify Databricks job. "
        f"HTTP {job_get_result['status_code']}: {job_get_result['response_text']}"
    )

deployed_job = job_get_result["job"]
deployed_settings = deployed_job.get("settings", {}) or {}

deployed_tasks = deployed_settings.get("tasks", []) or []
deployed_schedule = deployed_settings.get("schedule", {}) or {}
deployed_tags = deployed_settings.get("tags", {}) or {}

# ─────────────────────────────────────────────────────────────
# Product-readiness checks
# ─────────────────────────────────────────────────────────────

expected_task_count = len(serverless_job_payload.get("tasks", []))
actual_task_count = len(deployed_tasks)

tasks_with_classic_cluster = 0

for task in deployed_tasks:
    if (
        "job_cluster_key" in task
        or "new_cluster" in task
        or "existing_cluster_id" in task
    ):
        tasks_with_classic_cluster += 1

checks = {
    "job_exists": True,
    "job_name_matches": deployed_settings.get("name") == serverless_job_payload.get("name"),
    "task_count_matches": actual_task_count == expected_task_count,
    "schedule_exists": bool(deployed_schedule),
    "schedule_is_paused": deployed_schedule.get("pause_status") == "PAUSED",
    "timezone_matches": deployed_schedule.get("timezone_id") == JOB_TIMEZONE_ID,
    "cron_matches": deployed_schedule.get("quartz_cron_expression") == JOB_QUARTZ_CRON,
    "max_concurrent_runs_is_one": deployed_settings.get("max_concurrent_runs") == 1,
    "serverless_payload_confirmed": tasks_with_classic_cluster == 0,
    "project_tag_present": deployed_tags.get("project") == "supplysage_ai"
}

passed_checks = sum(1 for value in checks.values() if value is True)
failed_checks = sum(1 for value in checks.values() if value is not True)

overall_deployment_verified = failed_checks == 0

if overall_deployment_verified:
    final_product_status = "lakeflow_job_deployed_verified_paused"
else:
    final_product_status = "lakeflow_job_deployed_with_warnings"

verify_finished_at = datetime.now(timezone.utc)

job_url = f"{DATABRICKS_WORKSPACE_URL}/jobs/{JOB_ID}"

deployment_summary = {
    "workflow_deployment_id": workflow_deployment_id,
    "project_name": PROJECT_NAME,
    "workflow_name": WORKFLOW_NAME,
    "workflow_display_name": WORKFLOW_DISPLAY_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_id": JOB_ID,
    "job_name": deployed_settings.get("name"),
    "job_url": job_url,
    "expected_task_count": expected_task_count,
    "actual_task_count": actual_task_count,
    "schedule_quartz_cron": deployed_schedule.get("quartz_cron_expression"),
    "schedule_timezone_id": deployed_schedule.get("timezone_id"),
    "schedule_pause_status": deployed_schedule.get("pause_status"),
    "compute_mode": "serverless",
    "tasks_with_classic_cluster": tasks_with_classic_cluster,
    "max_concurrent_runs": deployed_settings.get("max_concurrent_runs"),
    "passed_checks": passed_checks,
    "failed_checks": failed_checks,
    "overall_deployment_verified": overall_deployment_verified,
    "final_product_status": final_product_status,
    "verification_checks": checks,
    "verified_at_utc": verify_finished_at.isoformat()
}

print("Lakeflow Job verification summary:")
print(json.dumps(deployment_summary, indent=2))

# ─────────────────────────────────────────────────────────────
# Save final deployment summary to Delta
# ─────────────────────────────────────────────────────────────

summary_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("project_name", StringType(), True),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_display_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("job_name", StringType(), True),
    StructField("job_url", StringType(), True),
    StructField("expected_task_count", IntegerType(), True),
    StructField("actual_task_count", IntegerType(), True),
    StructField("schedule_quartz_cron", StringType(), True),
    StructField("schedule_timezone_id", StringType(), True),
    StructField("schedule_pause_status", StringType(), True),
    StructField("compute_mode", StringType(), True),
    StructField("tasks_with_classic_cluster", IntegerType(), True),
    StructField("max_concurrent_runs", IntegerType(), True),
    StructField("passed_checks", IntegerType(), True),
    StructField("failed_checks", IntegerType(), True),
    StructField("overall_deployment_verified", BooleanType(), True),
    StructField("final_product_status", StringType(), True),
    StructField("verification_checks_json", StringType(), True),
    StructField("deployed_job_settings_json", StringType(), True),
    StructField("verified_at", TimestampType(), True),
])

summary_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "project_name": PROJECT_NAME,
    "workflow_name": WORKFLOW_NAME,
    "workflow_display_name": WORKFLOW_DISPLAY_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_id": JOB_ID,
    "job_name": deployed_settings.get("name"),
    "job_url": job_url,
    "expected_task_count": int(expected_task_count),
    "actual_task_count": int(actual_task_count),
    "schedule_quartz_cron": deployed_schedule.get("quartz_cron_expression"),
    "schedule_timezone_id": deployed_schedule.get("timezone_id"),
    "schedule_pause_status": deployed_schedule.get("pause_status"),
    "compute_mode": "serverless",
    "tasks_with_classic_cluster": int(tasks_with_classic_cluster),
    "max_concurrent_runs": int(deployed_settings.get("max_concurrent_runs") or 0),
    "passed_checks": int(passed_checks),
    "failed_checks": int(failed_checks),
    "overall_deployment_verified": bool(overall_deployment_verified),
    "final_product_status": final_product_status,
    "verification_checks_json": json.dumps(checks, indent=2, default=str),
    "deployed_job_settings_json": json.dumps(deployed_settings, indent=2, default=str),
    "verified_at": verify_finished_at
}]

deployment_summary_df = spark.createDataFrame(
    summary_rows,
    schema=summary_schema
)

(
    deployment_summary_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(WORKFLOW_DEPLOYMENT_SUMMARY_TABLE)
)

print(f"\nSaved final workflow deployment summary to: {WORKFLOW_DEPLOYMENT_SUMMARY_TABLE}")

# ─────────────────────────────────────────────────────────────
# Update main manifest status
# ─────────────────────────────────────────────────────────────

deployment_final_update_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("deployment_status", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("job_created_or_updated", BooleanType(), True),
])

deployment_final_update_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "deployment_status": final_product_status,
    "job_id": JOB_ID,
    "job_created_or_updated": True
}]

deployment_final_update_df = spark.createDataFrame(
    deployment_final_update_rows,
    schema=deployment_final_update_schema
)

deployment_final_update_df.createOrReplaceTempView("workflow_final_deployment_update")

spark.sql(f"""
MERGE INTO {WORKFLOW_JOB_MANIFEST_TABLE} AS target
USING workflow_final_deployment_update AS source
ON target.workflow_deployment_id = source.workflow_deployment_id
WHEN MATCHED THEN UPDATE SET
  target.deployment_status = source.deployment_status,
  target.job_id = source.job_id,
  target.job_created_or_updated = source.job_created_or_updated
""")

print(f"Updated workflow manifest final status: {WORKFLOW_JOB_MANIFEST_TABLE}")

display(
    spark.table(WORKFLOW_DEPLOYMENT_SUMMARY_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_id",
        "job_name",
        "expected_task_count",
        "actual_task_count",
        "schedule_pause_status",
        "compute_mode",
        "passed_checks",
        "failed_checks",
        "overall_deployment_verified",
        "final_product_status",
        "verified_at"
    )
    .orderBy(F.desc("verified_at"))
)

print("\nCell 7 complete.")

if overall_deployment_verified:
    print("Lakeflow deployment is verified and product-ready in PAUSED state.")
    print(f"Job URL: {job_url}")
else:
    print("Lakeflow deployment completed with warnings. Review failed checks above.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 8
# Unpause Lakeflow Job schedule and verify
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    TimestampType
)

assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."
assert "workflow_deployment_id" in globals(), "workflow_deployment_id missing. Rerun Cell 1."
assert "JOB_QUARTZ_CRON" in globals(), "JOB_QUARTZ_CRON missing. Rerun Cell 1."
assert "JOB_TIMEZONE_ID" in globals(), "JOB_TIMEZONE_ID missing. Rerun Cell 1."

# Use JOB_ID from Cell 7 if available, otherwise use deployment_result.
if "JOB_ID" in globals() and JOB_ID:
    target_job_id = str(JOB_ID)
elif "deployment_result" in globals() and deployment_result.get("job_id"):
    target_job_id = str(deployment_result.get("job_id"))
else:
    raise ValueError("No job ID found. Rerun Cell 6A and Cell 7 first.")

UNPAUSE_JOB_NOW = True

if not UNPAUSE_JOB_NOW:
    raise ValueError("UNPAUSE_JOB_NOW is False. Set it to True only when ready to unpause the schedule.")

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

unpause_started_at = datetime.now(timezone.utc)

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def get_databricks_job(job_id: str) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/get"
    params = {"job_id": job_id}

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=60
    )

    if response.status_code != 200:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:3000],
            "job": None
        }

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": response.text[:3000],
        "job": response.json()
    }


def update_job_schedule_pause_status(job_id: str, pause_status: str) -> dict:
    """
    Try Jobs API 2.2 first, then fall back to 2.1 if needed.
    """
    update_payload = {
        "job_id": int(job_id),
        "new_settings": {
            "schedule": {
                "quartz_cron_expression": JOB_QUARTZ_CRON,
                "timezone_id": JOB_TIMEZONE_ID,
                "pause_status": pause_status
            }
        }
    }

    api_versions = ["2.2", "2.1"]

    last_response = None

    for api_version in api_versions:
        url = f"{DATABRICKS_WORKSPACE_URL}/api/{api_version}/jobs/update"

        response = requests.post(
            url,
            headers=headers,
            data=json.dumps(update_payload),
            timeout=120
        )

        last_response = response

        if response.status_code in [200, 201]:
            return {
                "ok": True,
                "api_version": api_version,
                "status_code": response.status_code,
                "response_text": response.text[:3000]
            }

        # If endpoint/version unsupported, try fallback version.
        if response.status_code in [404, 405]:
            continue

        # Other errors are real errors.
        break

    return {
        "ok": False,
        "api_version": None,
        "status_code": last_response.status_code if last_response is not None else None,
        "response_text": last_response.text[:3000] if last_response is not None else "No response"
    }


# ─────────────────────────────────────────────────────────────
# Before state
# ─────────────────────────────────────────────────────────────

before_get = get_databricks_job(target_job_id)

if not before_get["ok"]:
    raise ValueError(
        f"Could not fetch job before unpause. "
        f"HTTP {before_get['status_code']}: {before_get['response_text']}"
    )

before_job = before_get["job"]
before_settings = before_job.get("settings", {}) or {}
before_schedule = before_settings.get("schedule", {}) or {}

print("Before unpause:")
print(json.dumps({
    "job_id": target_job_id,
    "job_name": before_settings.get("name"),
    "pause_status": before_schedule.get("pause_status"),
    "quartz_cron_expression": before_schedule.get("quartz_cron_expression"),
    "timezone_id": before_schedule.get("timezone_id")
}, indent=2))

# ─────────────────────────────────────────────────────────────
# Unpause schedule
# ─────────────────────────────────────────────────────────────

unpause_api_result = update_job_schedule_pause_status(
    job_id=target_job_id,
    pause_status="UNPAUSED"
)

unpause_finished_at = datetime.now(timezone.utc)

print("\nUnpause API result:")
print(json.dumps(unpause_api_result, indent=2))

if not unpause_api_result["ok"]:
    raise ValueError(
        f"Failed to unpause job schedule. "
        f"HTTP {unpause_api_result['status_code']}: {unpause_api_result['response_text']}"
    )

# ─────────────────────────────────────────────────────────────
# Verify after state
# ─────────────────────────────────────────────────────────────

after_get = get_databricks_job(target_job_id)

if not after_get["ok"]:
    raise ValueError(
        f"Could not fetch job after unpause. "
        f"HTTP {after_get['status_code']}: {after_get['response_text']}"
    )

after_job = after_get["job"]
after_settings = after_job.get("settings", {}) or {}
after_schedule = after_settings.get("schedule", {}) or {}

schedule_unpaused = after_schedule.get("pause_status") == "UNPAUSED"
cron_matches = after_schedule.get("quartz_cron_expression") == JOB_QUARTZ_CRON
timezone_matches = after_schedule.get("timezone_id") == JOB_TIMEZONE_ID

unpause_verified = bool(schedule_unpaused and cron_matches and timezone_matches)

if unpause_verified:
    final_schedule_status = "lakeflow_job_schedule_unpaused_verified"
else:
    final_schedule_status = "lakeflow_job_schedule_unpaused_with_warnings"

unpause_result = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": target_job_id,
    "job_name": after_settings.get("name"),
    "job_url": f"{DATABRICKS_WORKSPACE_URL}/jobs/{target_job_id}",
    "before_pause_status": before_schedule.get("pause_status"),
    "after_pause_status": after_schedule.get("pause_status"),
    "schedule_quartz_cron": after_schedule.get("quartz_cron_expression"),
    "schedule_timezone_id": after_schedule.get("timezone_id"),
    "schedule_unpaused": schedule_unpaused,
    "cron_matches": cron_matches,
    "timezone_matches": timezone_matches,
    "unpause_verified": unpause_verified,
    "final_schedule_status": final_schedule_status,
    "api_version_used": unpause_api_result.get("api_version"),
    "status_code": unpause_api_result.get("status_code"),
    "unpause_started_at": unpause_started_at.isoformat(),
    "unpause_finished_at": unpause_finished_at.isoformat(),
    "unpause_duration_seconds": (unpause_finished_at - unpause_started_at).total_seconds()
}

print("\nAfter unpause verification:")
print(json.dumps(unpause_result, indent=2))

# ─────────────────────────────────────────────────────────────
# Save unpause event
# ─────────────────────────────────────────────────────────────

UNPAUSE_EVENTS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_schedule_events"

unpause_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("job_name", StringType(), True),
    StructField("job_url", StringType(), True),
    StructField("before_pause_status", StringType(), True),
    StructField("after_pause_status", StringType(), True),
    StructField("schedule_quartz_cron", StringType(), True),
    StructField("schedule_timezone_id", StringType(), True),
    StructField("schedule_unpaused", BooleanType(), True),
    StructField("cron_matches", BooleanType(), True),
    StructField("timezone_matches", BooleanType(), True),
    StructField("unpause_verified", BooleanType(), True),
    StructField("final_schedule_status", StringType(), True),
    StructField("api_version_used", StringType(), True),
    StructField("status_code", IntegerType(), True),
    StructField("unpause_result_json", StringType(), True),
    StructField("unpause_started_at", TimestampType(), True),
    StructField("unpause_finished_at", TimestampType(), True),
])

unpause_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "workflow_name": WORKFLOW_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_id": target_job_id,
    "job_name": after_settings.get("name"),
    "job_url": f"{DATABRICKS_WORKSPACE_URL}/jobs/{target_job_id}",
    "before_pause_status": before_schedule.get("pause_status"),
    "after_pause_status": after_schedule.get("pause_status"),
    "schedule_quartz_cron": after_schedule.get("quartz_cron_expression"),
    "schedule_timezone_id": after_schedule.get("timezone_id"),
    "schedule_unpaused": bool(schedule_unpaused),
    "cron_matches": bool(cron_matches),
    "timezone_matches": bool(timezone_matches),
    "unpause_verified": bool(unpause_verified),
    "final_schedule_status": final_schedule_status,
    "api_version_used": unpause_api_result.get("api_version"),
    "status_code": int(unpause_api_result.get("status_code")) if unpause_api_result.get("status_code") is not None else None,
    "unpause_result_json": json.dumps(unpause_result, indent=2, default=str),
    "unpause_started_at": unpause_started_at,
    "unpause_finished_at": unpause_finished_at
}]

unpause_df = spark.createDataFrame(unpause_rows, schema=unpause_schema)

(
    unpause_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(UNPAUSE_EVENTS_TABLE)
)

print(f"\nSaved schedule event to: {UNPAUSE_EVENTS_TABLE}")

# ─────────────────────────────────────────────────────────────
# Update deployment summary and manifest final status
# ─────────────────────────────────────────────────────────────

schedule_update_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("schedule_pause_status", StringType(), True),
    StructField("final_product_status", StringType(), True),
])

schedule_update_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "schedule_pause_status": after_schedule.get("pause_status"),
    "final_product_status": final_schedule_status
}]

schedule_update_df = spark.createDataFrame(
    schedule_update_rows,
    schema=schedule_update_schema
)

schedule_update_df.createOrReplaceTempView("workflow_schedule_update")

spark.sql(f"""
MERGE INTO {WORKFLOW_DEPLOYMENT_SUMMARY_TABLE} AS target
USING workflow_schedule_update AS source
ON target.workflow_deployment_id = source.workflow_deployment_id
WHEN MATCHED THEN UPDATE SET
  target.schedule_pause_status = source.schedule_pause_status,
  target.final_product_status = source.final_product_status
""")

spark.sql(f"""
MERGE INTO {WORKFLOW_JOB_MANIFEST_TABLE} AS target
USING workflow_schedule_update AS source
ON target.workflow_deployment_id = source.workflow_deployment_id
WHEN MATCHED THEN UPDATE SET
  target.schedule_pause_status = source.schedule_pause_status,
  target.deployment_status = source.final_product_status
""")

print(f"Updated deployment summary: {WORKFLOW_DEPLOYMENT_SUMMARY_TABLE}")
print(f"Updated job manifest: {WORKFLOW_JOB_MANIFEST_TABLE}")

display(
    spark.table(UNPAUSE_EVENTS_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_id",
        "job_name",
        "before_pause_status",
        "after_pause_status",
        "schedule_quartz_cron",
        "schedule_timezone_id",
        "unpause_verified",
        "final_schedule_status",
        "unpause_finished_at"
    )
    .orderBy(F.desc("unpause_finished_at"))
)

if unpause_verified:
    print("\nLakeflow Job schedule is now UNPAUSED and verified.")
    print("SupplySage AI is scheduled to run daily at 6:00 AM America/Chicago.")
else:
    print("\nSchedule update completed with warnings. Review verification output above.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 9
# Trigger a manual Lakeflow Job run now
# ─────────────────────────────────────────────────────────────

import json
import uuid
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    LongType,
    TimestampType
)

assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."
assert "workflow_deployment_id" in globals(), "workflow_deployment_id missing. Rerun Cell 1."

# Use JOB_ID from Cell 7/8 if available, otherwise use deployment_result.
if "JOB_ID" in globals() and JOB_ID:
    target_job_id = str(JOB_ID)
elif "deployment_result" in globals() and deployment_result.get("job_id"):
    target_job_id = str(deployment_result.get("job_id"))
else:
    raise ValueError("No job ID found. Rerun Cell 6A and Cell 7 first.")

# This is intentionally explicit because this starts the full 26-task workflow.
TRIGGER_MANUAL_RUN_NOW = True

if not TRIGGER_MANUAL_RUN_NOW:
    raise ValueError(
        "TRIGGER_MANUAL_RUN_NOW is False. "
        "Set it to True only when you are ready to start the full Lakeflow Job."
    )

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

manual_run_started_at = datetime.now(timezone.utc)

idempotency_token = (
    f"supplysage_manual_run_"
    f"{workflow_deployment_id}_"
    f"{manual_run_started_at.strftime('%Y%m%d%H%M%S')}_"
    f"{uuid.uuid4().hex[:8]}"
)

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def trigger_job_run_now(job_id: str) -> dict:
    """
    Triggers a Databricks Job run immediately.
    Uses Jobs API 2.2 first, then falls back to 2.1.
    """
    payload = {
        "job_id": int(job_id),
        "idempotency_token": idempotency_token
    }

    api_versions = ["2.2", "2.1"]
    last_response = None

    for api_version in api_versions:
        url = f"{DATABRICKS_WORKSPACE_URL}/api/{api_version}/jobs/run-now"

        response = requests.post(
            url,
            headers=headers,
            data=json.dumps(payload),
            timeout=120
        )

        last_response = response

        if response.status_code in [200, 201]:
            result = response.json()
            return {
                "ok": True,
                "api_version": api_version,
                "status_code": response.status_code,
                "response_text": json.dumps(result, indent=2),
                "run_id": str(result.get("run_id")) if result.get("run_id") is not None else None,
                "number_in_job": int(result.get("number_in_job")) if result.get("number_in_job") is not None else None
            }

        if response.status_code in [404, 405]:
            continue

        break

    return {
        "ok": False,
        "api_version": None,
        "status_code": last_response.status_code if last_response is not None else None,
        "response_text": last_response.text[:3000] if last_response is not None else "No response",
        "run_id": None,
        "number_in_job": None
    }


def get_job_run(run_id: str) -> dict:
    """
    Fetches initial run state after triggering.
    """
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/runs/get"
    params = {"run_id": run_id}

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=60
    )

    if response.status_code != 200:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:3000],
            "run": None
        }

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": response.text[:3000],
        "run": response.json()
    }


# ─────────────────────────────────────────────────────────────
# Trigger manual run
# ─────────────────────────────────────────────────────────────

run_now_result = trigger_job_run_now(target_job_id)
manual_run_triggered_at = datetime.now(timezone.utc)

print("Manual Lakeflow run trigger result:")
print(json.dumps(run_now_result, indent=2))

if not run_now_result["ok"]:
    raise ValueError(
        f"Manual run trigger failed. "
        f"HTTP {run_now_result['status_code']}: {run_now_result['response_text']}"
    )

RUN_ID = run_now_result["run_id"]

if RUN_ID is None:
    raise ValueError("Run was triggered but no run_id was returned.")

# ─────────────────────────────────────────────────────────────
# Fetch initial run status
# ─────────────────────────────────────────────────────────────

run_get_result = get_job_run(RUN_ID)

if not run_get_result["ok"]:
    raise ValueError(
        f"Could not fetch run status. "
        f"HTTP {run_get_result['status_code']}: {run_get_result['response_text']}"
    )

run_payload = run_get_result["run"]
run_state = run_payload.get("state", {}) or {}

life_cycle_state = run_state.get("life_cycle_state")
result_state = run_state.get("result_state")
state_message = run_state.get("state_message")

run_page_url = run_payload.get("run_page_url") or f"{DATABRICKS_WORKSPACE_URL}/jobs/{target_job_id}/runs/{RUN_ID}"

manual_run_result = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": target_job_id,
    "run_id": RUN_ID,
    "number_in_job": run_now_result.get("number_in_job"),
    "run_page_url": run_page_url,
    "idempotency_token": idempotency_token,
    "trigger_api_version": run_now_result.get("api_version"),
    "trigger_status_code": run_now_result.get("status_code"),
    "trigger_succeeded": bool(run_now_result.get("ok")),
    "initial_life_cycle_state": life_cycle_state,
    "initial_result_state": result_state,
    "initial_state_message": state_message,
    "manual_run_started_at": manual_run_started_at.isoformat(),
    "manual_run_triggered_at": manual_run_triggered_at.isoformat()
}

print("\nInitial manual run status:")
print(json.dumps(manual_run_result, indent=2))

# ─────────────────────────────────────────────────────────────
# Save manual run event
# ─────────────────────────────────────────────────────────────

MANUAL_RUN_EVENTS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_manual_run_events"

manual_run_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("number_in_job", LongType(), True),
    StructField("run_page_url", StringType(), True),
    StructField("idempotency_token", StringType(), True),
    StructField("trigger_api_version", StringType(), True),
    StructField("trigger_status_code", IntegerType(), True),
    StructField("trigger_succeeded", BooleanType(), True),
    StructField("initial_life_cycle_state", StringType(), True),
    StructField("initial_result_state", StringType(), True),
    StructField("initial_state_message", StringType(), True),
    StructField("manual_run_result_json", StringType(), True),
    StructField("manual_run_started_at", TimestampType(), True),
    StructField("manual_run_triggered_at", TimestampType(), True),
])

manual_run_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "workflow_name": WORKFLOW_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_id": target_job_id,
    "run_id": RUN_ID,
    "number_in_job": run_now_result.get("number_in_job"),
    "run_page_url": run_page_url,
    "idempotency_token": idempotency_token,
    "trigger_api_version": run_now_result.get("api_version"),
    "trigger_status_code": int(run_now_result.get("status_code")) if run_now_result.get("status_code") is not None else None,
    "trigger_succeeded": bool(run_now_result.get("ok")),
    "initial_life_cycle_state": life_cycle_state,
    "initial_result_state": result_state,
    "initial_state_message": state_message,
    "manual_run_result_json": json.dumps(manual_run_result, indent=2, default=str),
    "manual_run_started_at": manual_run_started_at,
    "manual_run_triggered_at": manual_run_triggered_at
}]

manual_run_df = spark.createDataFrame(
    manual_run_rows,
    schema=manual_run_schema
)

(
    manual_run_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(MANUAL_RUN_EVENTS_TABLE)
)

print(f"\nSaved manual run event to: {MANUAL_RUN_EVENTS_TABLE}")

display(
    spark.table(MANUAL_RUN_EVENTS_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_id",
        "run_id",
        "trigger_succeeded",
        "initial_life_cycle_state",
        "initial_result_state",
        "initial_state_message",
        "run_page_url",
        "manual_run_triggered_at"
    )
    .orderBy(F.desc("manual_run_triggered_at"))
)

print("\nManual Lakeflow Job run has been triggered.")
print(f"Run ID: {RUN_ID}")
print(f"Run URL: {run_page_url}")
print("Use the Databricks Jobs UI to watch all 26 tasks complete.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 9A
# Trigger manual Lakeflow Job run with <=64 char idempotency token
# ─────────────────────────────────────────────────────────────

import json
import uuid
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    LongType,
    TimestampType
)

assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."
assert "workflow_deployment_id" in globals(), "workflow_deployment_id missing. Rerun Cell 1."

if "JOB_ID" in globals() and JOB_ID:
    target_job_id = str(JOB_ID)
elif "deployment_result" in globals() and deployment_result.get("job_id"):
    target_job_id = str(deployment_result.get("job_id"))
else:
    raise ValueError("No job ID found. Rerun Cell 6A and Cell 7 first.")

TRIGGER_MANUAL_RUN_NOW = True

if not TRIGGER_MANUAL_RUN_NOW:
    raise ValueError("TRIGGER_MANUAL_RUN_NOW is False.")

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

manual_run_started_at = datetime.now(timezone.utc)

# Must be <=64 chars.
idempotency_token = f"ss_{manual_run_started_at.strftime('%Y%m%d%H%M%S')}_{uuid.uuid4().hex[:8]}"

print("Idempotency token:", idempotency_token)
print("Token length:", len(idempotency_token))

assert len(idempotency_token) <= 64, "Idempotency token is too long."

# ─────────────────────────────────────────────────────────────
# Helpers
# ─────────────────────────────────────────────────────────────

def trigger_job_run_now(job_id: str) -> dict:
    payload = {
        "job_id": int(job_id),
        "idempotency_token": idempotency_token
    }

    for api_version in ["2.2", "2.1"]:
        url = f"{DATABRICKS_WORKSPACE_URL}/api/{api_version}/jobs/run-now"

        response = requests.post(
            url,
            headers=headers,
            data=json.dumps(payload),
            timeout=120
        )

        if response.status_code in [200, 201]:
            result = response.json()

            return {
                "ok": True,
                "api_version": api_version,
                "status_code": response.status_code,
                "response_text": json.dumps(result, indent=2),
                "run_id": str(result.get("run_id")) if result.get("run_id") is not None else None,
                "number_in_job": int(result.get("number_in_job")) if result.get("number_in_job") is not None else None
            }

        if response.status_code in [404, 405]:
            continue

        return {
            "ok": False,
            "api_version": api_version,
            "status_code": response.status_code,
            "response_text": response.text[:3000],
            "run_id": None,
            "number_in_job": None
        }

    return {
        "ok": False,
        "api_version": None,
        "status_code": None,
        "response_text": "No supported Jobs API version succeeded.",
        "run_id": None,
        "number_in_job": None
    }


def get_job_run(run_id: str) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/runs/get"
    params = {"run_id": run_id}

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=60
    )

    if response.status_code != 200:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:3000],
            "run": None
        }

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": response.text[:3000],
        "run": response.json()
    }


# ─────────────────────────────────────────────────────────────
# Trigger run
# ─────────────────────────────────────────────────────────────

run_now_result = trigger_job_run_now(target_job_id)
manual_run_triggered_at = datetime.now(timezone.utc)

print("\nManual Lakeflow run trigger result:")
print(json.dumps(run_now_result, indent=2))

if not run_now_result["ok"]:
    raise ValueError(
        f"Manual run trigger failed. "
        f"HTTP {run_now_result['status_code']}: {run_now_result['response_text']}"
    )

RUN_ID = run_now_result["run_id"]

if RUN_ID is None:
    raise ValueError("Run was triggered but no run_id was returned.")

# ─────────────────────────────────────────────────────────────
# Fetch initial run state
# ─────────────────────────────────────────────────────────────

run_get_result = get_job_run(RUN_ID)

if not run_get_result["ok"]:
    raise ValueError(
        f"Could not fetch run status. "
        f"HTTP {run_get_result['status_code']}: {run_get_result['response_text']}"
    )

run_payload = run_get_result["run"]
run_state = run_payload.get("state", {}) or {}

life_cycle_state = run_state.get("life_cycle_state")
result_state = run_state.get("result_state")
state_message = run_state.get("state_message")

run_page_url = (
    run_payload.get("run_page_url")
    or f"{DATABRICKS_WORKSPACE_URL}/jobs/{target_job_id}/runs/{RUN_ID}"
)

manual_run_result = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": target_job_id,
    "run_id": RUN_ID,
    "number_in_job": run_now_result.get("number_in_job"),
    "run_page_url": run_page_url,
    "idempotency_token": idempotency_token,
    "idempotency_token_length": len(idempotency_token),
    "trigger_api_version": run_now_result.get("api_version"),
    "trigger_status_code": run_now_result.get("status_code"),
    "trigger_succeeded": bool(run_now_result.get("ok")),
    "initial_life_cycle_state": life_cycle_state,
    "initial_result_state": result_state,
    "initial_state_message": state_message,
    "manual_run_started_at": manual_run_started_at.isoformat(),
    "manual_run_triggered_at": manual_run_triggered_at.isoformat()
}

print("\nInitial manual run status:")
print(json.dumps(manual_run_result, indent=2))

# ─────────────────────────────────────────────────────────────
# Save manual run event
# ─────────────────────────────────────────────────────────────

MANUAL_RUN_EVENTS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_manual_run_events"

manual_run_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("workflow_name", StringType(), True),
    StructField("workflow_version", StringType(), True),
    StructField("job_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("number_in_job", LongType(), True),
    StructField("run_page_url", StringType(), True),
    StructField("idempotency_token", StringType(), True),
    StructField("trigger_api_version", StringType(), True),
    StructField("trigger_status_code", IntegerType(), True),
    StructField("trigger_succeeded", BooleanType(), True),
    StructField("initial_life_cycle_state", StringType(), True),
    StructField("initial_result_state", StringType(), True),
    StructField("initial_state_message", StringType(), True),
    StructField("manual_run_result_json", StringType(), True),
    StructField("manual_run_started_at", TimestampType(), True),
    StructField("manual_run_triggered_at", TimestampType(), True),
])

manual_run_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "workflow_name": WORKFLOW_NAME,
    "workflow_version": WORKFLOW_VERSION,
    "job_id": target_job_id,
    "run_id": RUN_ID,
    "number_in_job": run_now_result.get("number_in_job"),
    "run_page_url": run_page_url,
    "idempotency_token": idempotency_token,
    "trigger_api_version": run_now_result.get("api_version"),
    "trigger_status_code": int(run_now_result.get("status_code")) if run_now_result.get("status_code") is not None else None,
    "trigger_succeeded": bool(run_now_result.get("ok")),
    "initial_life_cycle_state": life_cycle_state,
    "initial_result_state": result_state,
    "initial_state_message": state_message,
    "manual_run_result_json": json.dumps(manual_run_result, indent=2, default=str),
    "manual_run_started_at": manual_run_started_at,
    "manual_run_triggered_at": manual_run_triggered_at
}]

manual_run_df = spark.createDataFrame(
    manual_run_rows,
    schema=manual_run_schema
)

(
    manual_run_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(MANUAL_RUN_EVENTS_TABLE)
)

print(f"\nSaved manual run event to: {MANUAL_RUN_EVENTS_TABLE}")

display(
    spark.table(MANUAL_RUN_EVENTS_TABLE)
    .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
    .select(
        "workflow_deployment_id",
        "job_id",
        "run_id",
        "trigger_succeeded",
        "initial_life_cycle_state",
        "initial_result_state",
        "initial_state_message",
        "run_page_url",
        "manual_run_triggered_at"
    )
    .orderBy(F.desc("manual_run_triggered_at"))
)

print("\nManual Lakeflow Job run has been triggered.")
print(f"Run ID: {RUN_ID}")
print(f"Run URL: {run_page_url}")
print("Watch the Databricks Jobs UI for all 26 tasks.")

In [0]:
# ─────────────────────────────────────────────────────────────
# Notebook 30 — Cell 10
# Check latest manual Lakeflow Job run status
# ─────────────────────────────────────────────────────────────

import json
import requests
from datetime import datetime, timezone

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    BooleanType,
    IntegerType,
    TimestampType
)

assert "DATABRICKS_WORKSPACE_URL" in globals(), "DATABRICKS_WORKSPACE_URL missing. Rerun Cell 1."
assert "DATABRICKS_API_TOKEN" in globals(), "DATABRICKS_API_TOKEN missing. Rerun Cell 1."

if "RUN_ID" in globals() and RUN_ID:
    target_run_id = str(RUN_ID)
else:
    # Fallback to latest manual run event saved in Delta
    latest_run_row = (
        spark.table(f"{GOLD_SCHEMA}.gold_workflow_job_manual_run_events")
        .filter(F.col("workflow_deployment_id") == workflow_deployment_id)
        .orderBy(F.col("manual_run_triggered_at").desc())
        .select("run_id")
        .first()
    )

    if latest_run_row is None:
        raise ValueError("No RUN_ID found. Trigger a manual run first.")

    target_run_id = str(latest_run_row["run_id"])

headers = {
    "Authorization": f"Bearer {DATABRICKS_API_TOKEN}",
    "Content-Type": "application/json"
}

status_checked_at = datetime.now(timezone.utc)

def get_job_run(run_id: str) -> dict:
    url = f"{DATABRICKS_WORKSPACE_URL}/api/2.1/jobs/runs/get"
    params = {"run_id": run_id}

    response = requests.get(
        url,
        headers=headers,
        params=params,
        timeout=60
    )

    if response.status_code != 200:
        return {
            "ok": False,
            "status_code": response.status_code,
            "response_text": response.text[:3000],
            "run": None
        }

    return {
        "ok": True,
        "status_code": response.status_code,
        "response_text": response.text[:3000],
        "run": response.json()
    }


run_get_result = get_job_run(target_run_id)

if not run_get_result["ok"]:
    raise ValueError(
        f"Could not fetch run status. "
        f"HTTP {run_get_result['status_code']}: {run_get_result['response_text']}"
    )

run_payload = run_get_result["run"]
run_state = run_payload.get("state", {}) or {}
tasks = run_payload.get("tasks", []) or []

life_cycle_state = run_state.get("life_cycle_state")
result_state = run_state.get("result_state")
state_message = run_state.get("state_message")

run_page_url = run_payload.get("run_page_url")
job_id = str(run_payload.get("job_id")) if run_payload.get("job_id") is not None else None

task_status_rows = []

for task in tasks:
    task_state = task.get("state", {}) or {}

    task_status_rows.append({
        "run_id": target_run_id,
        "job_id": job_id,
        "task_key": task.get("task_key"),
        "run_page_url": task.get("run_page_url"),
        "life_cycle_state": task_state.get("life_cycle_state"),
        "result_state": task_state.get("result_state"),
        "state_message": task_state.get("state_message"),
        "start_time": task.get("start_time"),
        "end_time": task.get("end_time"),
        "status_checked_at": status_checked_at
    })

total_tasks = len(task_status_rows)
running_tasks = sum(1 for row in task_status_rows if row["life_cycle_state"] == "RUNNING")
pending_tasks = sum(1 for row in task_status_rows if row["life_cycle_state"] == "PENDING")
terminated_tasks = sum(1 for row in task_status_rows if row["life_cycle_state"] == "TERMINATED")
skipped_tasks = sum(1 for row in task_status_rows if row["life_cycle_state"] == "SKIPPED")
failed_tasks = sum(1 for row in task_status_rows if row["result_state"] in ["FAILED", "TIMEDOUT", "CANCELED"])

run_status_summary = {
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": job_id,
    "run_id": target_run_id,
    "run_page_url": run_page_url,
    "life_cycle_state": life_cycle_state,
    "result_state": result_state,
    "state_message": state_message,
    "total_tasks": total_tasks,
    "running_tasks": running_tasks,
    "pending_tasks": pending_tasks,
    "terminated_tasks": terminated_tasks,
    "skipped_tasks": skipped_tasks,
    "failed_tasks": failed_tasks,
    "status_checked_at": status_checked_at.isoformat()
}

print("Lakeflow manual run status summary:")
print(json.dumps(run_status_summary, indent=2))

if len(task_status_rows) > 0:
    task_status_schema = StructType([
        StructField("run_id", StringType(), True),
        StructField("job_id", StringType(), True),
        StructField("task_key", StringType(), True),
        StructField("run_page_url", StringType(), True),
        StructField("life_cycle_state", StringType(), True),
        StructField("result_state", StringType(), True),
        StructField("state_message", StringType(), True),
        StructField("start_time", StringType(), True),
        StructField("end_time", StringType(), True),
        StructField("status_checked_at", TimestampType(), True),
    ])

    task_status_df = spark.createDataFrame(
        task_status_rows,
        schema=task_status_schema
    )

    display(
        task_status_df.select(
            "task_key",
            "life_cycle_state",
            "result_state",
            "state_message",
            "run_page_url"
        )
    )

# Save run status snapshot
RUN_STATUS_TABLE = f"{GOLD_SCHEMA}.gold_workflow_job_run_status_snapshots"

run_status_schema = StructType([
    StructField("workflow_deployment_id", StringType(), False),
    StructField("job_id", StringType(), True),
    StructField("run_id", StringType(), True),
    StructField("run_page_url", StringType(), True),
    StructField("life_cycle_state", StringType(), True),
    StructField("result_state", StringType(), True),
    StructField("state_message", StringType(), True),
    StructField("total_tasks", IntegerType(), True),
    StructField("running_tasks", IntegerType(), True),
    StructField("pending_tasks", IntegerType(), True),
    StructField("terminated_tasks", IntegerType(), True),
    StructField("skipped_tasks", IntegerType(), True),
    StructField("failed_tasks", IntegerType(), True),
    StructField("run_status_summary_json", StringType(), True),
    StructField("status_checked_at", TimestampType(), True),
])

run_status_rows = [{
    "workflow_deployment_id": workflow_deployment_id,
    "job_id": job_id,
    "run_id": target_run_id,
    "run_page_url": run_page_url,
    "life_cycle_state": life_cycle_state,
    "result_state": result_state,
    "state_message": state_message,
    "total_tasks": int(total_tasks),
    "running_tasks": int(running_tasks),
    "pending_tasks": int(pending_tasks),
    "terminated_tasks": int(terminated_tasks),
    "skipped_tasks": int(skipped_tasks),
    "failed_tasks": int(failed_tasks),
    "run_status_summary_json": json.dumps(run_status_summary, indent=2, default=str),
    "status_checked_at": status_checked_at
}]

run_status_df = spark.createDataFrame(
    run_status_rows,
    schema=run_status_schema
)

(
    run_status_df
    .withColumn("gold_created_at", F.current_timestamp())
    .withColumn("gold_source_notebook", F.lit(NOTEBOOK_NAME))
    .write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .saveAsTable(RUN_STATUS_TABLE)
)

print(f"\nSaved run status snapshot to: {RUN_STATUS_TABLE}")

if life_cycle_state == "TERMINATED" and result_state == "SUCCESS":
    print("\nFull Lakeflow run completed successfully.")
elif failed_tasks > 0 or result_state in ["FAILED", "TIMEDOUT", "CANCELED"]:
    print("\nLakeflow run has failures. Check failed task details in the Jobs UI.")
else:
    print("\nLakeflow run is still in progress.")
    print(f"Run URL: {run_page_url}")